# RAF-DB test: three no-YOLO restoration paths

This notebook runs only three new MA3D paths on the full 3,068-image RAF-DB test split: RealESRGAN + CodeFormer, RealESRGAN only, and CodeFormer only. The RAF-DB inputs are already aligned 100x100 face crops, so these paths do not run YOLO or crop faces. Existing POSTER, MA3D baseline, and older YOLO-based results are left untouched and are not rerun.

It partitions the test images across both Kaggle T4 GPUs, reports errors by processing stage, counts incorrect predictions, and exports up to 10 high-confidence misclassified original images per path into three separate folders. Results are written to a new `results/predata_no_yolo/` directory.

Before running:

1. Upload the contents of `kaggle/dataset/` as a **private Kaggle Dataset**. Keep `bundle/` and `scripts/` together at its root, then attach that Dataset as notebook input.
2. Import this notebook file separately into Kaggle (`File > Import Notebook`) and select **GPU T4 x2**. Enable Internet for package/source and gated EMOCA asset setup.
3. Register and personally accept FLAME and EMOCA terms before running the asset-download cell. The consent flag now defaults to `True` as you requested.
4. Run from top to bottom. The first pass installs the pinned EMOCA environment and can take a while. A two-image smoke run exercises only the three new paths before the full test starts.

The notebook calls EMOCA on the same Kaggle runtime over loopback. The bundle does not include FLAME/EMOCA assets because those must be obtained by a registered user after accepting their terms. Kaggle usage sends the RAF-DB test data and included model artifacts to Kaggle's cloud storage/runtime; keep the dataset private. The notebook embeds the two new benchmark scripts, so the existing large Kaggle Dataset does not need a new version.

The environment setup accepts the Anaconda `pkgs/main` and `pkgs/r` channel terms, as authorized by the project owner. The FLAME/EMOCA consent flag defaults to `True`; the owner must still personally register and accept those terms before running the asset-download cell. The notebook downloads only the selected EMOCA checkpoint and runtime assets; it skips alternate checkpoints and example videos to reduce Kaggle disk usage.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

def run_quiet(command, *, label, env=None):
    completed = subprocess.run(
        command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors='replace',
    )
    if completed.returncode:
        tail = (completed.stdout or '').strip()[-4000:]
        raise RuntimeError(f'{label} failed (exit {completed.returncode}): {tail}')

# The project owner has already registered and accepted FLAME/EMOCA terms for this research run.
ACCEPT_FLAME_EMOCA_LICENSES = True
RUN_FULL_BENCHMARK = True
BUNDLE_ROOT_OVERRIDE = None  # Optional: exact path such as /kaggle/input/<dataset-slug>/bundle
WORKDIR = Path('/kaggle/working/rafdb5')

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
gpu_count = torch.cuda.device_count()
assert gpu_count == 2, f'This run is configured for T4 x2; found {gpu_count} visible GPU(s).'

if BUNDLE_ROOT_OVERRIDE:
    BUNDLE_ROOT = Path(BUNDLE_ROOT_OVERRIDE).resolve()
else:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('BUNDLE_MANIFEST.json')
                  if (p.parent / 'dataset' / 'test_labels.csv').is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one attached Kaggle bundle; found {candidates}. Set BUNDLE_ROOT_OVERRIDE.')
    BUNDLE_ROOT = candidates[0]
KAGGLE_DIR = BUNDLE_ROOT.parent
SCRIPTS = KAGGLE_DIR / 'scripts'
assert (SCRIPTS / 'prepare_runtime.py').is_file(), f'Bundle scripts missing beside {BUNDLE_ROOT}'


In [ ]:
# Keep Kaggle's preinstalled NumPy and CUDA PyTorch; the bundle pin is for isolated local environments.
requirements = BUNDLE_ROOT / 'environment' / 'requirements-worker.txt'
runtime_requirements = WORKDIR / 'requirements-worker-kaggle.txt'
WORKDIR.mkdir(parents=True, exist_ok=True)
requirements_lines = requirements.read_text(encoding='utf-8').splitlines()
runtime_requirements.write_text(
    '\n'.join(line for line in requirements_lines
                if not line.strip().lower().startswith('numpy==')) + '\n',
    encoding='utf-8',
)
run_quiet([sys.executable, '-m', 'pip', 'install', '-r', str(runtime_requirements)],
          label='Worker package installation')
import numpy as np
import cv2
run_quiet([sys.executable, str(SCRIPTS / 'verify_bundle.py'), str(BUNDLE_ROOT)],
          label='Bundle verification')


In [ ]:
# Stage project code and make sure the optional M6 package is complete in older uploaded Datasets.
run_quiet([sys.executable, str(SCRIPTS / 'prepare_runtime.py'),
           '--bundle-root', str(BUNDLE_ROOT), '--workdir', str(WORKDIR)],
          label='Runtime staging')
import base64
m6_pipeline = WORKDIR / 'project' / 'services' / 'image_preprocessing' / 'pipeline.py'
m6_pipeline.write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRpbWUgaW1wb3J0IHBlcmZfY291bnRlcgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBQcm90b2NvbAppbXBvcnQgbWF0aAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZU9wcwoKY2xhc3MgRGV0ZWN0b3IoUHJvdG9jb2wpOgogICAgbW9kZWxfaWQ6IHN0cgogICAgZGVmIGRldGVjdChzZWxmLCBpbWFnZTogSW1hZ2UuSW1hZ2UsIGNvbmZpZGVuY2U6IGZsb2F0KSAtPiBsaXN0W2RpY3Rbc3RyLCBBbnldXTogLi4uCmNsYXNzIFJlc3RvcmVyKFByb3RvY29sKToKICAgIG1vZGVsX2lkOiBzdHIKICAgIGRlZiByZXN0b3JlKHNlbGYsIGltYWdlOiBJbWFnZS5JbWFnZSwgZmlkZWxpdHk6IGZsb2F0IHwgTm9uZSA9IE5vbmUpIC0+IEltYWdlLkltYWdlOiAuLi4KCkBkYXRhY2xhc3MKY2xhc3MgUGlwZWxpbmVDb25maWc6CiAgICBjb25maWRlbmNlOiBmbG9hdCA9IDAuNQogICAgcGFkZGluZzogZmxvYXQgPSAwLjE1CiAgICBmaWRlbGl0eTogZmxvYXQgPSAwLjcKICAgIG1heF9zaWRlOiBpbnQgPSA0MDk2CiAgICBub19mYWNlX3BvbGljeTogc3RyID0gImtlZXAiCiAgICBudW1weV9jb2xvcjogc3RyID0gIlJHQiIKQGRhdGFjbGFzcwpjbGFzcyBGYWNlUmVzdWx0OgogICAgYmJveF94eXh5OiBsaXN0W2ludF0KICAgIGNyb3BfYmJveF94eXh5OiBsaXN0W2ludF0KICAgIGNvbmZpZGVuY2U6IGZsb2F0CiAgICBjcm9wOiBJbWFnZS5JbWFnZQpAZGF0YWNsYXNzCmNsYXNzIFBpcGVsaW5lUmVzdWx0OgogICAgZW5oYW5jZWQ6IEltYWdlLkltYWdlIHwgTm9uZQogICAgZmFjZXM6IGxpc3RbRmFjZVJlc3VsdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGlucHV0X2RpbWVuc2lvbnM6IGxpc3RbaW50XSB8IE5vbmUgPSBOb25lCiAgICBvdXRwdXRfZGltZW5zaW9uczogbGlzdFtpbnRdIHwgTm9uZSA9IE5vbmUKICAgIG1vZGVsX2lkczogZGljdFtzdHIsIHN0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHN0YWdlX3RpbWVzX21zOiBkaWN0W3N0ciwgZmxvYXRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBzdGF0dXM6IHN0ciA9ICJlcnJvciIKICAgIGVycm9yOiBzdHIgfCBOb25lID0gTm9uZQoKZGVmIF9pbnB1dF9yZ2IoaW1hZ2U6IHN0ciB8IFBhdGggfCBJbWFnZS5JbWFnZSB8IG5wLm5kYXJyYXksIG51bXB5X2NvbG9yOiBzdHIpIC0+IEltYWdlLkltYWdlOgogICAgaWYgaXNpbnN0YW5jZShpbWFnZSwgKHN0ciwgUGF0aCkpOgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWFnZSkgYXMgc3JjOiByZXR1cm4gSW1hZ2VPcHMuZXhpZl90cmFuc3Bvc2Uoc3JjKS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2UsIEltYWdlLkltYWdlKTogcmV0dXJuIEltYWdlT3BzLmV4aWZfdHJhbnNwb3NlKGltYWdlKS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2UsIG5wLm5kYXJyYXkpOgogICAgICAgIGlmIGltYWdlLmR0eXBlICE9IG5wLnVpbnQ4IG9yIGltYWdlLm5kaW0gIT0gMyBvciBpbWFnZS5zaGFwZVsyXSAhPSAzIG9yIG1pbihpbWFnZS5zaGFwZVs6Ml0pIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTnVtUHkgaW5wdXQgbXVzdCBiZSBub24tZW1wdHkgSHhXeDMgdWludDgiKQogICAgICAgIGlmIG51bXB5X2NvbG9yIG5vdCBpbiAoIlJHQiIsICJCR1IiKTogcmFpc2UgVmFsdWVFcnJvcigibnVtcHlfY29sb3IgbXVzdCBiZSBSR0Igb3IgQkdSIikKICAgICAgICBhcnIgPSBpbWFnZSBpZiBudW1weV9jb2xvciA9PSAiUkdCIiBlbHNlIGltYWdlWy4uLiwgOjotMV0KICAgICAgICByZXR1cm4gSW1hZ2UuZnJvbWFycmF5KG5wLmFzY29udGlndW91c2FycmF5KGFyciksICJSR0IiKQogICAgcmFpc2UgVHlwZUVycm9yKCJpbnB1dCBtdXN0IGJlIGEgcGF0aCwgUElMIGltYWdlLCBvciBOdW1QeSBSR0IgYXJyYXkiKQoKZGVmIF94eXh5KGJveDogQW55LCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCkgLT4gbGlzdFtpbnRdOgogICAgdiA9IG5wLmFzYXJyYXkoYm94LCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xKQogICAgaWYgbGVuKHYpICE9IDQgb3Igbm90IG5wLmlzZmluaXRlKHYpLmFsbCgpOiByYWlzZSBWYWx1ZUVycm9yKCJkZXRlY3RvciByZXR1cm5lZCBtYWxmb3JtZWQgYmJveF94eXh5IikKICAgIHgxLCB5MSwgeDIsIHkyID0gdgogICAgeDEsIHgyID0gc29ydGVkKChpbnQobnAuZmxvb3IoeDEpKSwgaW50KG5wLmNlaWwoeDIpKSkpCiAgICB5MSwgeTIgPSBzb3J0ZWQoKGludChucC5mbG9vcih5MSkpLCBpbnQobnAuY2VpbCh5MikpKSkKICAgIHgxLCB4MiA9IG1heCgwLCBtaW4od2lkdGgsIHgxKSksIG1heCgwLCBtaW4od2lkdGgsIHgyKSkKICAgIHkxLCB5MiA9IG1heCgwLCBtaW4oaGVpZ2h0LCB5MSkpLCBtYXgoMCwgbWluKGhlaWdodCwgeTIpKQogICAgaWYgeDIgPD0geDEgb3IgeTIgPD0geTE6IHJhaXNlIFZhbHVlRXJyb3IoImRldGVjdG9yIHJldHVybmVkIGVtcHR5IGJib3giKQogICAgcmV0dXJuIFt4MSwgeTEsIHgyLCB5Ml0KCmRlZiBfcGFkZGVkKGJveDogbGlzdFtpbnRdLCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCwgcGFkZGluZzogZmxvYXQpIC0+IGxpc3RbaW50XToKICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHBhZGRpbmcpIG9yIHBhZGRpbmcgPCAwOiByYWlzZSBWYWx1ZUVycm9yKCJwYWRkaW5nIG11c3QgYmUgZmluaXRlIGFuZCBub24tbmVnYXRpdmUiKQogICAgeDEseTEseDIseTIgPSBib3g7IGJ3LGJoPXgyLXgxLHkyLXkxOyBjeCxjeT0oeDEreDIpLzIsKHkxK3kyKS8yCiAgICByZXR1cm4gW21heCgwLGludChucC5mbG9vcihjeC1idyooLjUrcGFkZGluZykpKSksIG1heCgwLGludChucC5mbG9vcihjeS1iaCooLjUrcGFkZGluZykpKSksCiAgICAgICAgICAgIG1pbih3aWR0aCxpbnQobnAuY2VpbChjeCtidyooLjUrcGFkZGluZykpKSksIG1pbihoZWlnaHQsaW50KG5wLmNlaWwoY3krYmgqKC41K3BhZGRpbmcpKSkpXQoKZGVmIF9saW1pdChpbWFnZTogSW1hZ2UuSW1hZ2UsIG1heF9zaWRlOiBpbnQpIC0+IEltYWdlLkltYWdlOgogICAgaWYgbWF4X3NpZGUgPCAxOiByYWlzZSBWYWx1ZUVycm9yKCJtYXhfc2lkZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHcsaD1pbWFnZS5zaXplOyBzY2FsZT1taW4oMS4sbWF4X3NpZGUvbWF4KHcsaCkpCiAgICByZXR1cm4gaW1hZ2UgaWYgc2NhbGUgPT0gMSBlbHNlIGltYWdlLnJlc2l6ZSgobWF4KDEsaW50KHcqc2NhbGUpKSxtYXgoMSxpbnQoaCpzY2FsZSkpKSxJbWFnZS5SZXNhbXBsaW5nLkxBTkNaT1MpCgpjbGFzcyBJbWFnZVBpcGVsaW5lOiAgIyBzdGFuZGFsb25lIE02IHByb2Nlc3Npbmcgb3JjaGVzdHJhdGlvbgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGdlbmVyYWxfcmVzdG9yZXI6IFJlc3RvcmVyLCBkZXRlY3RvcjogRGV0ZWN0b3IsIGZhY2VfcmVzdG9yZXI6IFJlc3RvcmVyLCBjb25maWc6IFBpcGVsaW5lQ29uZmlnfE5vbmU9Tm9uZSk6CiAgICAgICAgc2VsZi5nZW5lcmFsX3Jlc3RvcmVyLHNlbGYuZGV0ZWN0b3Isc2VsZi5mYWNlX3Jlc3RvcmVyPWdlbmVyYWxfcmVzdG9yZXIsZGV0ZWN0b3IsZmFjZV9yZXN0b3JlcgogICAgICAgIHNlbGYuY29uZmlnPWNvbmZpZyBvciBQaXBlbGluZUNvbmZpZygpCiAgICAgICAgaWYgc2VsZi5jb25maWcubm9fZmFjZV9wb2xpY3kgbm90IGluICgia2VlcCIsImVycm9yIik6IHJhaXNlIFZhbHVlRXJyb3IoIm5vX2ZhY2VfcG9saWN5IG11c3QgYmUga2VlcCBvciBlcnJvciIpCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoc2VsZi5jb25maWcucGFkZGluZykgb3Igc2VsZi5jb25maWcucGFkZGluZyA8IDA6IHJhaXNlIFZhbHVlRXJyb3IoInBhZGRpbmcgbXVzdCBiZSBmaW5pdGUgYW5kIG5vbi1uZWdhdGl2ZSIpCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoc2VsZi5jb25maWcuY29uZmlkZW5jZSkgb3Igbm90IDAgPD0gc2VsZi5jb25maWcuY29uZmlkZW5jZSA8PSAxOiByYWlzZSBWYWx1ZUVycm9yKCJjb25maWRlbmNlIG11c3QgYmUgaW4gWzAsIDFdIikKICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShzZWxmLmNvbmZpZy5maWRlbGl0eSkgb3Igbm90IDAgPD0gc2VsZi5jb25maWcuZmlkZWxpdHkgPD0gMTogcmFpc2UgVmFsdWVFcnJvcigiZmlkZWxpdHkgbXVzdCBiZSBpbiBbMCwgMV0iKQogICAgICAgIGlmIHNlbGYuY29uZmlnLm1heF9zaWRlIDwgMTogcmFpc2UgVmFsdWVFcnJvcigibWF4X3NpZGUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgaWYgc2VsZi5jb25maWcubnVtcHlfY29sb3Igbm90IGluICgiUkdCIiwgIkJHUiIpOiByYWlzZSBWYWx1ZUVycm9yKCJudW1weV9jb2xvciBtdXN0IGJlIFJHQiBvciBCR1IiKQogICAgZGVmIHByb2Nlc3Moc2VsZiwgc291cmNlOiBzdHJ8UGF0aHxJbWFnZS5JbWFnZXxucC5uZGFycmF5KSAtPiBQaXBlbGluZVJlc3VsdDoKICAgICAgICByPVBpcGVsaW5lUmVzdWx0KGVuaGFuY2VkPU5vbmUsbW9kZWxfaWRzPXsiZ2VuZXJhbCI6c2VsZi5nZW5lcmFsX3Jlc3RvcmVyLm1vZGVsX2lkLCJkZXRlY3RvciI6c2VsZi5kZXRlY3Rvci5tb2RlbF9pZCwiZmFjZSI6c2VsZi5mYWNlX3Jlc3RvcmVyLm1vZGVsX2lkfSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9yaWdpbmFsPV9pbnB1dF9yZ2Ioc291cmNlLHNlbGYuY29uZmlnLm51bXB5X2NvbG9yKTsgci5pbnB1dF9kaW1lbnNpb25zPVtvcmlnaW5hbC53aWR0aCxvcmlnaW5hbC5oZWlnaHRdCiAgICAgICAgICAgIHQ9cGVyZl9jb3VudGVyKCk7IGVuaGFuY2VkPXNlbGYuZ2VuZXJhbF9yZXN0b3Jlci5yZXN0b3JlKG9yaWdpbmFsLmNvcHkoKSk7IHIuc3RhZ2VfdGltZXNfbXNbImdlbmVyYWxfcmVzdG9yYXRpb24iXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShlbmhhbmNlZCxJbWFnZS5JbWFnZSk6IHJhaXNlIFR5cGVFcnJvcigiZ2VuZXJhbCByZXN0b3JlciBtdXN0IHJldHVybiBQSUwuSW1hZ2UiKQogICAgICAgICAgICBlbmhhbmNlZD1lbmhhbmNlZC5jb252ZXJ0KCJSR0IiKQogICAgICAgICAgICBpZiBlbmhhbmNlZC5zaXplICE9IG9yaWdpbmFsLnNpemU6IHJhaXNlIFZhbHVlRXJyb3IoImdlbmVyYWwgcmVzdG9yYXRpb24gbXVzdCBwcmVzZXJ2ZSBkaW1lbnNpb25zOyBiYm94IG1hcHBpbmcgaXMgdW5kZWZpbmVkIikKICAgICAgICAgICAgci5lbmhhbmNlZD1lbmhhbmNlZDsgci5vdXRwdXRfZGltZW5zaW9ucz1bZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0XQogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpOyBkZXRlY3Rpb25zPXNlbGYuZGV0ZWN0b3IuZGV0ZWN0KGVuaGFuY2VkLmNvcHkoKSxzZWxmLmNvbmZpZy5jb25maWRlbmNlKTsgci5zdGFnZV90aW1lc19tc1siZmFjZV9kZXRlY3Rpb24iXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZvciBkIGluIGRldGVjdGlvbnM6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgY29uZmlkZW5jZT1mbG9hdChkWyJjb25maWRlbmNlIl0pCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoY29uZmlkZW5jZSkgb3Igbm90IDAgPD0gY29uZmlkZW5jZSA8PSAxOiByYWlzZSBWYWx1ZUVycm9yKCJkZXRlY3RvciBjb25maWRlbmNlIG11c3QgYmUgZmluaXRlIGFuZCBpbiBbMCwgMV0iKQogICAgICAgICAgICAgICAgICAgIGlmIGNvbmZpZGVuY2UgPCBzZWxmLmNvbmZpZy5jb25maWRlbmNlOiBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGJveD1feHl4eShkLmdldCgiYmJveF94eXh5IiksZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0KTsgY2I9X3BhZGRlZChib3gsZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0LHNlbGYuY29uZmlnLnBhZGRpbmcpCiAgICAgICAgICAgICAgICAgICAgci5mYWNlcy5hcHBlbmQoRmFjZVJlc3VsdChib3gsY2IsY29uZmlkZW5jZSxlbmhhbmNlZC5jcm9wKHR1cGxlKGNiKSkuY29udmVydCgiUkdCIikpKQogICAgICAgICAgICAgICAgZXhjZXB0IChLZXlFcnJvciwgVHlwZUVycm9yLCBWYWx1ZUVycm9yKSBhcyBleGM6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImRldGVjdG9yIHJldHVybmVkIG1hbGZvcm1lZCBmYWNlOiB7ZXhjfSIpIGZyb20gZXhjCiAgICAgICAgICAgIHIuc3RhZ2VfdGltZXNfbXNbImZhY2VfY3JvcHBpbmciXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZvciBmIGluIHIuZmFjZXM6CiAgICAgICAgICAgICAgICByZXN0b3JlZD1zZWxmLmZhY2VfcmVzdG9yZXIucmVzdG9yZShmLmNyb3Asc2VsZi5jb25maWcuZmlkZWxpdHkpCiAgICAgICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyZXN0b3JlZCxJbWFnZS5JbWFnZSk6IHJhaXNlIFR5cGVFcnJvcigiZmFjZSByZXN0b3JlciBtdXN0IHJldHVybiBQSUwuSW1hZ2UiKQogICAgICAgICAgICAgICAgZi5jcm9wPV9saW1pdChyZXN0b3JlZC5jb252ZXJ0KCJSR0IiKSxzZWxmLmNvbmZpZy5tYXhfc2lkZSkKICAgICAgICAgICAgci5zdGFnZV90aW1lc19tc1siZmFjZV9yZXN0b3JhdGlvbiJdPShwZXJmX2NvdW50ZXIoKS10KSoxMDAwCiAgICAgICAgICAgIHQ9cGVyZl9jb3VudGVyKCk7IHIuZW5oYW5jZWQ9X2xpbWl0KGVuaGFuY2VkLHNlbGYuY29uZmlnLm1heF9zaWRlKTsgci5vdXRwdXRfZGltZW5zaW9ucz1bci5lbmhhbmNlZC53aWR0aCxyLmVuaGFuY2VkLmhlaWdodF0KICAgICAgICAgICAgci5zdGFnZV90aW1lc19tc1sib3V0cHV0X3NpemVfbGltaXQiXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICBpZiBub3Qgci5mYWNlcyBhbmQgc2VsZi5jb25maWcubm9fZmFjZV9wb2xpY3kgPT0gImVycm9yIjogcmFpc2UgUnVudGltZUVycm9yKCJub19mYWNlIikKICAgICAgICAgICAgci5zdGF0dXM9Im9rIiBpZiByLmZhY2VzIGVsc2UgIm5vX2ZhY2UiCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiByLnN0YXR1cyxyLmVycm9yPSJlcnJvciIsZiJ7dHlwZShlKS5fX25hbWVfX306IHtlfSIKICAgICAgICByZXR1cm4gcgoKZGVmIHByb2Nlc3NfYmF0Y2gocGlwZWxpbmU6IEltYWdlUGlwZWxpbmUsIHNvdXJjZXMpOgogICAgIiIiUHJvY2VzcyBldmVyeSBpbnB1dCBpbmRlcGVuZGVudGx5OyBhIGJhZCBmaWxlIGRvZXMgbm90IHN0b3AgdGhlIGJhdGNoLiIiIgogICAgcmV0dXJuIFsoc291cmNlLCBwaXBlbGluZS5wcm9jZXNzKHNvdXJjZSkpIGZvciBzb3VyY2UgaW4gc291cmNlc10K'))
source_setup = BUNDLE_ROOT / 'environment' / 'setup_emoca_source.sh'
run_quiet(['bash', str(source_setup), str(BUNDLE_ROOT), str(WORKDIR)],
          label='EMOCA source setup')


In [ ]:
# Build the project's pinned Python 3.8 / Torch 1.11 / PyTorch3D EMOCA environment.
# Reuse Miniconda if a previous setup attempt already created it under /kaggle/working.
WORKDIR.mkdir(parents=True, exist_ok=True)
# Clear large unused assets from a previous upstream download before setup touches the disk.
asset_root = WORKDIR / 'project' / 'model' / 'emoca' / 'assets'
for relative in (
    'EMOCA/models/EMOCA.zip', 'EMOCA/models/EMOCA_v2_mp.zip',
    'EMOCA/models/EMOCA_v2_lr_cos_1.5.zip', 'data/EMOCA_test_example_data.zip',
    'DECA.zip', 'FaceRecognition.zip', 'FLAME.zip',
    'EMOCA/models/EMOCA', 'EMOCA/models/EMOCA_v2_mp',
    'EMOCA/models/EMOCA_v2_lr_cos_1.5', 'data/EMOCA_test_example_data',
):
    old_asset = asset_root / relative
    if old_asset.is_dir():
        shutil.rmtree(old_asset)
    elif old_asset.exists():
        old_asset.unlink()
try:
    (asset_root / 'data').rmdir()
except OSError:
    pass
miniconda = WORKDIR / 'miniconda'
miniconda_conda = miniconda / 'bin' / 'conda'
if miniconda_conda.is_file():
    os.environ['PATH'] = str(miniconda / 'bin') + os.pathsep + os.environ.get('PATH', '')
    os.environ['CONDA_EXE'] = str(miniconda_conda)
elif shutil.which('conda') is None:
    installer = WORKDIR / 'miniconda.sh'
    subprocess.run(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh', '-O', str(installer)], check=True)
    subprocess.run(['bash', str(installer), '-b', '-p', str(miniconda)], check=True)
    os.environ['PATH'] = str(miniconda / 'bin') + os.pathsep + os.environ.get('PATH', '')
    os.environ['CONDA_EXE'] = str(miniconda_conda)
else:
    os.environ.setdefault('CONDA_EXE', shutil.which('conda'))
env_setup = BUNDLE_ROOT / 'environment' / 'setup_emoca_env.sh'
env_setup_text = env_setup.read_text(encoding='utf-8')
# Avoid Kaggle's unrelated Python mamba test-runner command and cross-version PYTHONPATH.
env_setup_text = env_setup_text.replace(
    'eval "$(conda shell.bash hook)"',
    'CONDA_EXE="$(command -v conda)"\n'
    'CONDA_BIN="$(dirname "$CONDA_EXE")"\n'
    'export PATH="$CONDA_BIN:$PATH"\n'
    'unset PYTHONPATH\n'
    'eval "$("$CONDA_EXE" shell.bash hook)"',
)
env_setup_text = env_setup_text.replace(
    'if ! command -v mamba >/dev/null; then',
    'if [[ ! -x "$CONDA_BIN/mamba" ]]; then',
)
env_setup_text = env_setup_text.replace(
    '  conda install -n base -c conda-forge mamba=2.3.3 -y',
    '  "$CONDA_EXE" install -n base -c conda-forge mamba=2.3.3 -y',
)
env_setup_text = env_setup_text.replace(
    '  mamba create -n work38 python=3.8 -y',
    '  "$CONDA_BIN/mamba" create -n work38 python=3.8 -y',
)
env_setup_text = env_setup_text.replace(
    'mamba env update -n work38 --file',
    '"$CONDA_BIN/mamba" env update -n work38 --file',
)
# Apply package fixes even if this notebook is attached to an older uploaded Dataset.
setup_lines = []
has_numpy_pin = 'python -m pip install numpy==1.23.5' in env_setup_text
found_pip_upgrade = False
for line in env_setup_text.splitlines():
    if 'python -m pip install --upgrade pip==' in line:
        found_pip_upgrade = True
        setup_lines.append(line)
        if not has_numpy_pin:
            setup_lines.append('conda run -n work38 python -m pip install numpy==1.23.5')
        continue
    if line.rstrip().endswith('python -m pip install Cython==0.29'):
        setup_lines.append(line.replace('Cython==0.29', 'Cython==0.29.37'))
        continue
    if 'python -m pip install onnx==1.10.2 onnxruntime-gpu' in line:
        setup_lines.append(line.replace(
            'onnx==1.10.2 onnxruntime-gpu ', ''
        ).replace('mediapipe ', 'mediapipe==0.10.9 '))
    elif '^(onnx|youtube-dl|mediapipe|mmcv-full|pyzmq)' in line:
        setup_lines.append(line.replace(
            '^(onnx|youtube-dl|mediapipe|mmcv-full|pyzmq)',
            '^(onnx|onnxruntime-gpu|youtube-dl|mediapipe|mmcv-full|pyzmq|opencv-python|opencv-python-headless|opencv-contrib-python|opencv-contrib-python-headless|insightface|deepspeech-gpu)',
        ))
    elif 'python -m pip install mmcv-full -f ' in line:
        continue  # Not imported anywhere in this benchmark's inference path.
    else:
        setup_lines.append(line)
if not found_pip_upgrade and not has_numpy_pin:
    raise RuntimeError('Could not place the pinned NumPy before installing EMOCA requirements.')
env_setup_text = '\n'.join(setup_lines) + '\n'
if 'opencv-python-headless==4.5.5.64' not in env_setup_text:
    editable_install = 'conda run -n work38 python -m pip install -e "$EMOCA_ROOT"'
    if editable_install not in env_setup_text:
        raise RuntimeError('Could not find EMOCA editable-install line for OpenCV normalization.')
    opencv_normalization = (
        editable_install + '\n'
        'conda run -n work38 python -m pip uninstall -y opencv-python opencv-contrib-python opencv-contrib-python-headless opencv-python-headless || true\n'
        'conda run -n work38 python -m pip install opencv-python-headless==4.5.5.64\n'
        'conda run -n work38 python -c "import cv2, mediapipe; print(cv2.__version__, mediapipe.__version__)"\n'
    )
    env_setup_text = env_setup_text.replace(editable_install, opencv_normalization, 1)
if 'tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main' not in env_setup_text:
    conda_hook = 'eval "$("$CONDA_EXE" shell.bash hook)"'
    if conda_hook not in env_setup_text:
        raise RuntimeError('Could not locate Conda hook to add the authorized Anaconda ToS acceptance.')
    tos_acceptance = (
        conda_hook + '\n'
        '# User-authorized acceptance for the Anaconda channels listed by EMOCA.\n'
        '"$CONDA_EXE" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main\n'
        '"$CONDA_EXE" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r'
    )
    env_setup_text = env_setup_text.replace(conda_hook, tos_acceptance, 1)
if 'mamba create -n work38' in env_setup_text:
    raise RuntimeError('Could not patch the bundled Mamba command for Kaggle.')
env_setup_runtime = WORKDIR / 'setup_emoca_env_kaggle.sh'
env_setup_runtime.write_text(env_setup_text, encoding='utf-8')
setup_env = os.environ.copy()
setup_env['CONDA_EXE'] = setup_env.get('CONDA_EXE') or shutil.which('conda') or ''
if not setup_env['CONDA_EXE']:
    raise RuntimeError('Conda executable not found after Miniconda setup.')
setup_env['PATH'] = str(Path(setup_env['CONDA_EXE']).parent) + os.pathsep + setup_env.get('PATH', '')
setup_env.pop('PYTHONPATH', None)
os.environ.pop('PYTHONPATH', None)
run_quiet([setup_env['CONDA_EXE'], 'clean', '--all', '--yes'],
          env=setup_env, label='Conda cache cleanup')
run_quiet(['bash', str(env_setup_runtime), str(BUNDLE_ROOT), str(WORKDIR)],
          env=setup_env, label='EMOCA environment setup')


In [ ]:
# License-gated download: fetch only benchmark assets, not upstream demos/alternate checkpoints.
if not ACCEPT_FLAME_EMOCA_LICENSES:
    raise RuntimeError('Asset download is disabled. Enable it only after personally accepting FLAME and EMOCA terms.')
import base64
asset_env = os.environ.copy()
asset_env['ACCEPT_FLAME_EMOCA_LICENSES'] = '1'
asset_setup = WORKDIR / 'download_emoca_assets_minimal.sh'
asset_setup.write_bytes(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgYmFzaApzZXQgLWV1byBwaXBlZmFpbAoKV09SS0RJUj0iJHsxOj91c2FnZTogZG93bmxvYWRfZW1vY2FfYXNzZXRzLnNoIFdPUktESVJ9IgppZiBbWyAiJHtBQ0NFUFRfRkxBTUVfRU1PQ0FfTElDRU5TRVM6LTF9IiAhPSAiMSIgXV07IHRoZW4KICBjYXQgPiYyIDw8J0VPRicKQmxvY2tlZCBieSBBQ0NFUFRfRkxBTUVfRU1PQ0FfTElDRU5TRVM9MC4gU2V0IGl0IHRvIDEgb25seSBhZnRlciBwZXJzb25hbGx5IGFjY2VwdGluZyB0aGUgRkxBTUUgYW5kIEVNT0NBIHRlcm1zLgpFT0YKICBleGl0IDEwCmZpCgpFTU9DQV9ST09UPSIkV09SS0RJUi9wcm9qZWN0L21vZGVsL2Vtb2NhIgpBU1NFVF9ST09UPSIkRU1PQ0FfUk9PVC9hc3NldHMiCnRlc3QgLWQgIiRFTU9DQV9ST09UL2dkbF9hcHBzL0VNT0NBL2RlbW9zIiB8fCB7IGVjaG8gIlBpbm5lZCBFTU9DQSBzb3VyY2UgaXMgbWlzc2luZy4iID4mMjsgZXhpdCAyOyB9CmNvbW1hbmQgLXYgd2dldCA+L2Rldi9udWxsIHx8IHsgZWNobyAid2dldCBpcyByZXF1aXJlZC4iID4mMjsgZXhpdCAyOyB9CmNvbW1hbmQgLXYgdW56aXAgPi9kZXYvbnVsbCB8fCB7IGVjaG8gInVuemlwIGlzIHJlcXVpcmVkLiIgPiYyOyBleGl0IDI7IH0KCiMgVGhlIGJlbmNobWFyayBvbmx5IGxvYWRzIEVNT0NBX3YyX2xyX21zZV8yMC4gUmVtb3ZlIHVudXNlZCBkZW1vIGRhdGEsCiMgbGVnYWN5L2FsdGVybmF0ZSBjaGVja3BvaW50cyBhbmQgc3RhbGUgYXJjaGl2ZXMgbGVmdCBieSB0aGUgZmFpbGVkIHJ1bi4Kcm0gLWYgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9FTU9DQS56aXAiIFwKICAiJEFTU0VUX1JPT1QvRU1PQ0EvbW9kZWxzL0VNT0NBX3YyX21wLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfY29zXzEuNS56aXAiIFwKICAiJEFTU0VUX1JPT1QvZGF0YS9FTU9DQV90ZXN0X2V4YW1wbGVfZGF0YS56aXAiIFwKICAiJEFTU0VUX1JPT1QvREVDQS56aXAiICIkQVNTRVRfUk9PVC9GYWNlUmVjb2duaXRpb24uemlwIiAiJEFTU0VUX1JPT1QvRkxBTUUuemlwIgpybSAtcmYgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9FTU9DQSIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbXAiIFwKICAiJEFTU0VUX1JPT1QvRU1PQ0EvbW9kZWxzL0VNT0NBX3YyX2xyX2Nvc18xLjUiIFwKICAiJEFTU0VUX1JPT1QvZGF0YS9FTU9DQV90ZXN0X2V4YW1wbGVfZGF0YSIKcm1kaXIgIiRBU1NFVF9ST09UL2RhdGEiIDI+L2Rldi9udWxsIHx8IHRydWUKCiMgUmVjbGFpbSBwYWNrYWdlIGNhY2hlcyBiZWZvcmUgZXh0cmFjdGluZyBhc3NldHMgaW50byBLYWdnbGUncyBsaW1pdGVkIGRpc2suCmlmIFtbIC1uICIke0NPTkRBX0VYRTotfSIgJiYgLXggIiR7Q09OREFfRVhFfSIgXV07IHRoZW4KICAiJENPTkRBX0VYRSIgY2xlYW4gLS1hbGwgLS15ZXMgPi9kZXYvbnVsbCAyPiYxIHx8IHRydWUKICAiJENPTkRBX0VYRSIgcnVuIC1uIHdvcmszOCBweXRob24gLW0gcGlwIGNhY2hlIHB1cmdlID4vZGV2L251bGwgMj4mMSB8fCB0cnVlCmZpCgpkb3dubG9hZF9leHRyYWN0KCkgewogIGxvY2FsIHVybD0iJDEiIGFyY2hpdmU9IiQyIiBkZXN0aW5hdGlvbj0iJDMiCiAgbWtkaXIgLXAgIiRkZXN0aW5hdGlvbiIgIiQoZGlybmFtZSAiJGFyY2hpdmUiKSIKICBybSAtZiAiJGFyY2hpdmUiICIkYXJjaGl2ZS5wYXJ0IgogIHdnZXQgLXEgIiR1cmwiIC1PICIkYXJjaGl2ZS5wYXJ0IgogIG12ICIkYXJjaGl2ZS5wYXJ0IiAiJGFyY2hpdmUiCiAgdW56aXAgLXEgLW8gIiRhcmNoaXZlIiAtZCAiJGRlc3RpbmF0aW9uIgogIHJtIC1mICIkYXJjaGl2ZSIKfQoKQkFTRV9VUkw9Imh0dHBzOi8vZG93bmxvYWQuaXMudHVlLm1wZy5kZS9lbW9jYS9hc3NldHMiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwLnppcCIgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscyIKZG93bmxvYWRfZXh0cmFjdCAiJEJBU0VfVVJML0VNT0NBL21vZGVscy9ERUNBLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvREVDQS56aXAiICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9ERUNBLnppcCIgIiRBU1NFVF9ST09UL0RFQ0EuemlwIiAiJEFTU0VUX1JPT1QiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9GYWNlUmVjb2duaXRpb24uemlwIiAiJEFTU0VUX1JPT1QvRmFjZVJlY29nbml0aW9uLnppcCIgIiRBU1NFVF9ST09UIgpkb3dubG9hZF9leHRyYWN0ICIkQkFTRV9VUkwvRkxBTUUuemlwIiAiJEFTU0VUX1JPT1QvRkxBTUUuemlwIiAiJEFTU0VUX1JPT1QiCgpmb3IgcmVxdWlyZWQgaW4gXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwIiBcCiAgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9ERUNBIiBcCiAgIiRBU1NFVF9ST09UL0RFQ0EiIFwKICAiJEFTU0VUX1JPT1QvRmFjZVJlY29nbml0aW9uIiBcCiAgIiRBU1NFVF9ST09UL0ZMQU1FIjsgZG8KICB0ZXN0IC1kICIkcmVxdWlyZWQiIHx8IHsgZWNobyAiUmVxdWlyZWQgRU1PQ0EgYXNzZXQgd2FzIG5vdCBleHRyYWN0ZWQ6ICRyZXF1aXJlZCIgPiYyOyBleGl0IDM7IH0KZG9uZQoKZWNobyAiUmVxdWlyZWQgRU1PQ0EgYXNzZXRzIHJlYWR5OyB1bnVzZWQgY2hlY2twb2ludHMgYW5kIGRlbW8gZGF0YSBza2lwcGVkLiIK'))
run_quiet(['bash', str(asset_setup), str(WORKDIR)], env=asset_env,
          label='EMOCA/FLAME asset download')


In [ ]:
# Install the new worker/summarizer into writable Kaggle storage; do not alter the attached Dataset.
import base64
PREPROCESS_SCRIPTS = WORKDIR / 'predata_no_yolo_scripts'
PREPROCESS_SCRIPTS.mkdir(parents=True, exist_ok=True)
(PREPROCESS_SCRIPTS / 'run_predata_worker.py').write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSBQSUwgaW1wb3J0IEltYWdlCgoKVkFSSUFOVFMgPSAoIk1BM0RSZUNvX25vWU9MTyIsICJNQTNEUmVfbm9ZT0xPIiwgIk1BM0RDb19ub1lPTE8iKQoKCmRlZiBpbWFnZV9ieXRlcyhpbWFnZTogSW1hZ2UuSW1hZ2UpIC0+IGJ5dGVzOgogICAgYnVmZmVyID0gaW8uQnl0ZXNJTygpCiAgICBpbWFnZS5jb252ZXJ0KCJSR0IiKS5zYXZlKGJ1ZmZlciwgZm9ybWF0PSJQTkciKQogICAgcmV0dXJuIGJ1ZmZlci5nZXR2YWx1ZSgpCgoKZGVmIGdldF9tYTNkX3ByZWRpY3Rpb24ocmVzdWx0OiBkaWN0KSAtPiBkaWN0OgogICAgZm9yIGZhY2UgaW4gcmVzdWx0LmdldCgiZmFjZXMiKSBvciBbXToKICAgICAgICBmb3IgcHJlZGljdGlvbiBpbiBmYWNlLmdldCgicHJlZGljdGlvbnMiKSBvciBbXToKICAgICAgICAgICAgaWYgcHJlZGljdGlvbi5nZXQoIm1vZGVsX2lkIikgIT0gIm1hM2RfcmFmZGIiOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgcHJlZGljdGlvbi5nZXQoInN0YXR1cyIpID09ICJvayI6CiAgICAgICAgICAgICAgICB0b3AgPSBwcmVkaWN0aW9uLmdldCgidG9wMSIpIG9yIHt9CiAgICAgICAgICAgICAgICBwcm9icyA9IFtmbG9hdCh2YWx1ZSkgZm9yIHZhbHVlIGluIHByZWRpY3Rpb24uZ2V0KCJwcm9icyIsIFtdKV0KICAgICAgICAgICAgICAgIGlmIGxlbihwcm9icykgIT0gNzoKICAgICAgICAgICAgICAgICAgICByZXR1cm4geyJvayI6IEZhbHNlLCAiZXJyb3IiOiBmImludmFsaWRfb3V0cHV0OiBleHBlY3RlZCA3IHByb2JhYmlsaXRpZXMsIGdvdCB7bGVuKHByb2JzKX0ifQogICAgICAgICAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgICAgICAgICAib2siOiBUcnVlLAogICAgICAgICAgICAgICAgICAgICJpbmRleCI6IGludCh0b3BbImluZGV4Il0pLAogICAgICAgICAgICAgICAgICAgICJjb25maWRlbmNlIjogZmxvYXQodG9wWyJwcm9iIl0pLAogICAgICAgICAgICAgICAgICAgICJwcm9icyI6IHByb2JzLAogICAgICAgICAgICAgICAgfQogICAgICAgICAgICBlcnJvciA9IHByZWRpY3Rpb24uZ2V0KCJlcnJvciIpIG9yIHt9CiAgICAgICAgICAgIHJldHVybiB7Im9rIjogRmFsc2UsICJlcnJvciI6IGYie2Vycm9yLmdldCgnY29kZScsICdwcmVkaWN0aW9uX2Vycm9yJyl9OiB7ZXJyb3IuZ2V0KCdtZXNzYWdlJywgJycpfSJ9CiAgICBlcnJvciA9IHJlc3VsdC5nZXQoImVycm9yIikgb3Ige30KICAgIHdhcm5pbmdzID0gcmVzdWx0LmdldCgid2FybmluZ3MiKSBvciBbXQogICAgcmV0dXJuIHsib2siOiBGYWxzZSwgImVycm9yIjogZiJ7ZXJyb3IuZ2V0KCdjb2RlJywgJ2FuYWx5emVfZXJyb3InKX06IHtlcnJvci5nZXQoJ21lc3NhZ2UnLCB3YXJuaW5ncyl9In0KCgpkZWYgc3RhZ2VfcmVjb3JkKHN0YXR1czogc3RyLCBzdGFydGVkOiBmbG9hdCwgZXJyb3I6IHN0ciB8IE5vbmUgPSBOb25lKSAtPiBkaWN0OgogICAgcmV0dXJuIHsKICAgICAgICAic3RhdHVzIjogc3RhdHVzLAogICAgICAgICJlcnJvciI6IGVycm9yLAogICAgICAgICJlbGFwc2VkX21zIjogcm91bmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAqIDEwMDAsIDIpLAogICAgfQoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IkV2YWx1YXRlIG9ubHkgdGhlIHRocmVlIG5vLVlPTE8gcHJlcHJvY2Vzc2luZyBicmFuY2hlcy4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1idW5kbGUtcm9vdCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcHJvamVjdC1yb290IiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1lbW9jYS11cmwiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXQtanNvbmwiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNoYXJkLWluZGV4IiwgdHlwZT1pbnQsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNoYXJkcyIsIHR5cGU9aW50LCBkZWZhdWx0PTIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWxpbWl0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgaGVscD0iU21va2UtdGVzdCBjYXA7IDAgbWVhbnMgdGhlIGNvbXBsZXRlIHRlc3Qgc2V0LiIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgIGJ1bmRsZSA9IGFyZ3MuYnVuZGxlX3Jvb3QucmVzb2x2ZSgpCiAgICBwcm9qZWN0ID0gYXJncy5wcm9qZWN0X3Jvb3QucmVzb2x2ZSgpCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgc3RyKHByb2plY3QpKQogICAgb3MuZW52aXJvblsiTUEzRF9ERVZJQ0UiXSA9ICJjdWRhOjAiIGlmIG9zLmVudmlyb24uZ2V0KCJDVURBX1ZJU0lCTEVfREVWSUNFUyIpIGlzIG5vdCBOb25lIGVsc2UgImF1dG8iCgogICAgZnJvbSBzZXJ2aWNlcy5pbWFnZV9wcmVwcm9jZXNzaW5nLmFkYXB0ZXJzIGltcG9ydCBDb2RlRm9ybWVyUmVzdG9yZXIsIFNwYW5kcmVsUmVzdG9yZXIKICAgIGZyb20gd2ViLm9yY2hlc3RyYXRvciBpbXBvcnQgYW5hbHl6ZSwgY2hlY2tfZW1vY2EKCiAgICBsYWJlbHNfcGF0aCA9IGJ1bmRsZSAvICJkYXRhc2V0IiAvICJ0ZXN0X2xhYmVscy5jc3YiCiAgICB3aXRoIGxhYmVsc19wYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04LXNpZyIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICBsYWJlbHMgPSB7cm93WyJpbWFnZSJdOiBpbnQocm93WyJsYWJlbCJdKSBmb3Igcm93IGluIGNzdi5EaWN0UmVhZGVyKHN0cmVhbSl9CiAgICBlbnRyaWVzID0gc29ydGVkKCgobGFiZWwsIG5hbWUpIGZvciBuYW1lLCBsYWJlbCBpbiBsYWJlbHMuaXRlbXMoKSksIGtleT1sYW1iZGEgaXRlbTogKGl0ZW1bMF0sIGl0ZW1bMV0pKQogICAgaWYgYXJncy5saW1pdDoKICAgICAgICBlbnRyaWVzID0gZW50cmllc1s6YXJncy5saW1pdF0KICAgIGVudHJpZXMgPSBbZW50cnkgZm9yIGluZGV4LCBlbnRyeSBpbiBlbnVtZXJhdGUoZW50cmllcykgaWYgaW5kZXggJSBhcmdzLnNoYXJkcyA9PSBhcmdzLnNoYXJkX2luZGV4XQoKICAgIGVtb2NhID0geyJtb2RlIjogInJlYWwiLCAidXJsIjogYXJncy5lbW9jYV91cmwsICJ0aW1lb3V0X3MiOiAxMjB9CiAgICBoZWFsdGggPSBjaGVja19lbW9jYShlbW9jYSkKICAgIGlmIG5vdCBoZWFsdGguZ2V0KCJvayIpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkVNT0NBIGlzIG5vdCBoZWFsdGh5OyByZWZ1c2luZyBtb2NrIGZhbGxiYWNrOiB7aGVhbHRofSIpCgogICAgZ2VuZXJhbF9wYXRoID0gcHJvamVjdCAvICJtb2RlbCIgLyAicmVhbGVzci1nZW5lcmFsLXg0djMiIC8gInJlYWxlc3ItZ2VuZXJhbC14NHYzLnB0aCIKICAgIGNvZGVmb3JtZXJfcGF0aCA9IHByb2plY3QgLyAibW9kZWwiIC8gImNvZGVmb3JtZXIiIC8gImNvZGVmb3JtZXIucHRoIgogICAgY29kZWZvcm1lcl9yb290ID0gcHJvamVjdCAvICJ0aGlyZF9wYXJ0eSIgLyAiQ29kZUZvcm1lciIKICAgIGdlbmVyYWwgPSBmYWNlID0gTm9uZQogICAgbG9hZF9lcnJvcnM6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIHRyeToKICAgICAgICBnZW5lcmFsID0gU3BhbmRyZWxSZXN0b3JlcihnZW5lcmFsX3BhdGgsICJSZWFsRVNSR0FOIENvbXBhY3QgKGxvY2FsIGNoZWNrcG9pbnQpIikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGxvYWRfZXJyb3JzWyJyZWFsZXNyZ2FuIl0gPSBmInt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIKICAgIHRyeToKICAgICAgICBmYWNlID0gQ29kZUZvcm1lclJlc3RvcmVyKGNvZGVmb3JtZXJfcGF0aCwgY29kZWZvcm1lcl9yb290KQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgbG9hZF9lcnJvcnNbImNvZGVmb3JtZXIiXSA9IGYie3R5cGUoZXhjKS5fX25hbWVfX306IHtleGN9IgoKICAgIG91dF9wYXRoID0gYXJncy5vdXRfanNvbmwucmVzb2x2ZSgpCiAgICBvdXRfcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgZG9uZTogc2V0W3R1cGxlW2ludCwgc3RyXV0gPSBzZXQoKQogICAgaWYgb3V0X3BhdGguaXNfZmlsZSgpOgogICAgICAgIHdpdGggb3V0X3BhdGgub3BlbihlbmNvZGluZz0idXRmLTgiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIGZvciBsaW5lIGluIHN0cmVhbToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICByb3cgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgICAgICAgICAgICAgcHJlZGljdGlvbnMgPSByb3cuZ2V0KCJ2YXJpYW50cyIsIHt9KQogICAgICAgICAgICAgICAgICAgIGlmIGFsbChwcmVkaWN0aW9ucy5nZXQobmFtZSwge30pLmdldCgib2siKSBmb3IgbmFtZSBpbiBWQVJJQU5UUyk6CiAgICAgICAgICAgICAgICAgICAgICAgIGRvbmUuYWRkKChpbnQocm93WyJsYWJlbCJdKSwgc3RyKHJvd1siZmlsZSJdKSkpCiAgICAgICAgICAgICAgICBleGNlcHQgKGpzb24uSlNPTkRlY29kZUVycm9yLCBLZXlFcnJvciwgVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgcGVuZGluZyA9IFtlbnRyeSBmb3IgZW50cnkgaW4gZW50cmllcyBpZiBlbnRyeSBub3QgaW4gZG9uZV0KICAgIHByaW50KAogICAgICAgIGYiU2hhcmQge2FyZ3Muc2hhcmRfaW5kZXh9L3thcmdzLnNoYXJkc306IGltYWdlcz17bGVuKGVudHJpZXMpfSBwZW5kaW5nPXtsZW4ocGVuZGluZyl9OyAiCiAgICAgICAgZiJSZWFsRVNSR0FOPXsncmVhZHknIGlmIGdlbmVyYWwgZWxzZSAndW5hdmFpbGFibGUnfTsgIgogICAgICAgIGYiQ29kZUZvcm1lcj17J3JlYWR5JyBpZiBmYWNlIGVsc2UgJ3VuYXZhaWxhYmxlJ307IEVNT0NBPXtlbW9jYVsndXJsJ119IiwKICAgICAgICBmbHVzaD1UcnVlLAogICAgKQoKICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICB3aXRoIG91dF9wYXRoLm9wZW4oImEiLCBlbmNvZGluZz0idXRmLTgiLCBidWZmZXJpbmc9MSkgYXMgc3RyZWFtOgogICAgICAgIGZvciBsb2NhbF9pbmRleCwgKGxhYmVsLCBmaWxlbmFtZSkgaW4gZW51bWVyYXRlKHBlbmRpbmcsIDEpOgogICAgICAgICAgICBpdGVtX3N0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGltYWdlX3BhdGggPSBidW5kbGUgLyAiZGF0YXNldCIgLyAiREFUQVNFVCIgLyAidGVzdCIgLyBzdHIobGFiZWwpIC8gZmlsZW5hbWUKICAgICAgICAgICAgcm93OiBkaWN0ID0gewogICAgICAgICAgICAgICAgImxhYmVsIjogbGFiZWwsCiAgICAgICAgICAgICAgICAiZmlsZSI6IGZpbGVuYW1lLAogICAgICAgICAgICAgICAgImRpbWVuc2lvbnMiOiBOb25lLAogICAgICAgICAgICAgICAgInZhcmlhbnRzIjoge30sCiAgICAgICAgICAgICAgICAic3RhZ2VzIjoge30sCiAgICAgICAgICAgICAgICAiZW1vY2FfcmV0cmllcyI6IDAsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgd2l0aCBJbWFnZS5vcGVuKGltYWdlX3BhdGgpIGFzIHNvdXJjZToKICAgICAgICAgICAgICAgICAgICBvcmlnaW5hbCA9IHNvdXJjZS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgICAgICAgICAgICAgIHJvd1siZGltZW5zaW9ucyJdID0gbGlzdChvcmlnaW5hbC5zaXplKQoKICAgICAgICAgICAgICAgICMgUnVuIGVhY2ggcmVzdG9yZXIgb24gdGhlIGZ1bGwsIGFscmVhZHkgYWxpZ25lZCBSQUYtREIgZmFjZSBpbWFnZS4KICAgICAgICAgICAgICAgICMgTm8gWU9MTyBtb2RlbCwgZmFjZSBkZXRlY3Rvciwgb3IgZGV0ZWN0b3IgY3JvcCBpcyB1c2VkIGhlcmUuCiAgICAgICAgICAgICAgICByZWFsX2ltYWdlID0gTm9uZQogICAgICAgICAgICAgICAgc3RhZ2Vfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgICAgIGlmIGdlbmVyYWwgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByb3dbInN0YWdlcyJdWyJyZWFsZXNyZ2FuIl0gPSBzdGFnZV9yZWNvcmQoCiAgICAgICAgICAgICAgICAgICAgICAgICJlcnJvciIsIHN0YWdlX3N0YXJ0ZWQsIGxvYWRfZXJyb3JzLmdldCgicmVhbGVzcmdhbiIsICJSZWFsRVNSR0FOIHVuYXZhaWxhYmxlIikKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICAgICAgcmVhbF9pbWFnZSA9IGdlbmVyYWwucmVzdG9yZShvcmlnaW5hbC5jb3B5KCkpCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJlYWxfaW1hZ2UsIEltYWdlLkltYWdlKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFR5cGVFcnJvcigiUmVhbEVTUkdBTiBtdXN0IHJldHVybiBhIFBJTCBpbWFnZSIpCiAgICAgICAgICAgICAgICAgICAgICAgIHJlYWxfaW1hZ2UgPSByZWFsX2ltYWdlLmNvbnZlcnQoIlJHQiIpCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIHJlYWxfaW1hZ2Uuc2l6ZSAhPSBvcmlnaW5hbC5zaXplOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlJlYWxFU1JHQU4gY2hhbmdlZCBpbWFnZSBzaXplIHtvcmlnaW5hbC5zaXplfSAtPiB7cmVhbF9pbWFnZS5zaXplfSIpCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bInJlYWxlc3JnYW4iXSA9IHN0YWdlX3JlY29yZCgib2siLCBzdGFnZV9zdGFydGVkKQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICAgICAgICAgICAgICByb3dbInN0YWdlcyJdWyJyZWFsZXNyZ2FuIl0gPSBzdGFnZV9yZWNvcmQoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiLCBzdGFnZV9zdGFydGVkLCBmInt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIKICAgICAgICAgICAgICAgICAgICAgICAgKQoKICAgICAgICAgICAgICAgIHJhd19jb2RlZm9ybWVyID0gTm9uZQogICAgICAgICAgICAgICAgc3RhZ2Vfc3RhcnRlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICAgICAgICAgIGlmIGZhY2UgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByb3dbInN0YWdlcyJdWyJjb2RlZm9ybWVyX3JhdyJdID0gc3RhZ2VfcmVjb3JkKAogICAgICAgICAgICAgICAgICAgICAgICAiZXJyb3IiLCBzdGFnZV9zdGFydGVkLCBsb2FkX2Vycm9ycy5nZXQoImNvZGVmb3JtZXIiLCAiQ29kZUZvcm1lciB1bmF2YWlsYWJsZSIpCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhd19jb2RlZm9ybWVyID0gZmFjZS5yZXN0b3JlKG9yaWdpbmFsLmNvcHkoKSkKICAgICAgICAgICAgICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UocmF3X2NvZGVmb3JtZXIsIEltYWdlLkltYWdlKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFR5cGVFcnJvcigiQ29kZUZvcm1lciBtdXN0IHJldHVybiBhIFBJTCBpbWFnZSIpCiAgICAgICAgICAgICAgICAgICAgICAgIHJhd19jb2RlZm9ybWVyID0gcmF3X2NvZGVmb3JtZXIuY29udmVydCgiUkdCIikKICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJzdGFnZXMiXVsiY29kZWZvcm1lcl9yYXciXSA9IHN0YWdlX3JlY29yZCgib2siLCBzdGFnZV9zdGFydGVkKQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICAgICAgICAgICAgICByb3dbInN0YWdlcyJdWyJjb2RlZm9ybWVyX3JhdyJdID0gc3RhZ2VfcmVjb3JkKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgImVycm9yIiwgc3RhZ2Vfc3RhcnRlZCwgZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCiAgICAgICAgICAgICAgICAgICAgICAgICkKCiAgICAgICAgICAgICAgICByZWFsX2NvZGVmb3JtZXIgPSBOb25lCiAgICAgICAgICAgICAgICBzdGFnZV9zdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICAgICAgaWYgcmVhbF9pbWFnZSBpcyBOb25lOgogICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bImNvZGVmb3JtZXJfYWZ0ZXJfcmVhbGVzcmdhbiJdID0gc3RhZ2VfcmVjb3JkKAogICAgICAgICAgICAgICAgICAgICAgICAic2tpcHBlZCIsIHN0YWdlX3N0YXJ0ZWQsICJSZWFsRVNSR0FOIHN0YWdlIGZhaWxlZCIKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBlbGlmIGZhY2UgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByb3dbInN0YWdlcyJdWyJjb2RlZm9ybWVyX2FmdGVyX3JlYWxlc3JnYW4iXSA9IHN0YWdlX3JlY29yZCgKICAgICAgICAgICAgICAgICAgICAgICAgImVycm9yIiwgc3RhZ2Vfc3RhcnRlZCwgbG9hZF9lcnJvcnMuZ2V0KCJjb2RlZm9ybWVyIiwgIkNvZGVGb3JtZXIgdW5hdmFpbGFibGUiKQogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICByZWFsX2NvZGVmb3JtZXIgPSBmYWNlLnJlc3RvcmUocmVhbF9pbWFnZS5jb3B5KCkpCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJlYWxfY29kZWZvcm1lciwgSW1hZ2UuSW1hZ2UpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgcmFpc2UgVHlwZUVycm9yKCJDb2RlRm9ybWVyIG11c3QgcmV0dXJuIGEgUElMIGltYWdlIikKICAgICAgICAgICAgICAgICAgICAgICAgcmVhbF9jb2RlZm9ybWVyID0gcmVhbF9jb2RlZm9ybWVyLmNvbnZlcnQoIlJHQiIpCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bImNvZGVmb3JtZXJfYWZ0ZXJfcmVhbGVzcmdhbiJdID0gc3RhZ2VfcmVjb3JkKCJvayIsIHN0YWdlX3N0YXJ0ZWQpCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bImNvZGVmb3JtZXJfYWZ0ZXJfcmVhbGVzcmdhbiJdID0gc3RhZ2VfcmVjb3JkKAogICAgICAgICAgICAgICAgICAgICAgICAgICAgImVycm9yIiwgc3RhZ2Vfc3RhcnRlZCwgZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCiAgICAgICAgICAgICAgICAgICAgICAgICkKCiAgICAgICAgICAgICAgICBicmFuY2hfaW1hZ2VzID0gewogICAgICAgICAgICAgICAgICAgICJNQTNEUmVDb19ub1lPTE8iOiByZWFsX2NvZGVmb3JtZXIsCiAgICAgICAgICAgICAgICAgICAgIk1BM0RSZV9ub1lPTE8iOiByZWFsX2ltYWdlLAogICAgICAgICAgICAgICAgICAgICJNQTNEQ29fbm9ZT0xPIjogcmF3X2NvZGVmb3JtZXIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICBicmFuY2hfc3RhZ2VzID0gewogICAgICAgICAgICAgICAgICAgICJNQTNEUmVDb19ub1lPTE8iOiAiZW1vY2FfbWEzZF9yZWFsZXNyZ2FuX2NvZGVmb3JtZXIiLAogICAgICAgICAgICAgICAgICAgICJNQTNEUmVfbm9ZT0xPIjogImVtb2NhX21hM2RfcmVhbGVzcmdhbiIsCiAgICAgICAgICAgICAgICAgICAgIk1BM0RDb19ub1lPTE8iOiAiZW1vY2FfbWEzZF9jb2RlZm9ybWVyIiwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGZvciB2YXJpYW50IGluIFZBUklBTlRTOgogICAgICAgICAgICAgICAgICAgIGJyYW5jaF9pbWFnZSA9IGJyYW5jaF9pbWFnZXNbdmFyaWFudF0KICAgICAgICAgICAgICAgICAgICBzdGFnZV9uYW1lID0gYnJhbmNoX3N0YWdlc1t2YXJpYW50XQogICAgICAgICAgICAgICAgICAgIHN0YWdlX3N0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgICAgICAgICAgICAgaWYgYnJhbmNoX2ltYWdlIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sidmFyaWFudHMiXVt2YXJpYW50XSA9IHsKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJvayI6IEZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgImVycm9yIjogInByZXByb2Nlc3NpbmdfdW5hdmFpbGFibGU6IHJlcXVpcmVkIHJlc3RvcmF0aW9uIHN0YWdlIGZhaWxlZCIsCiAgICAgICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJzdGFnZXMiXVtzdGFnZV9uYW1lXSA9IHN0YWdlX3JlY29yZCgKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJza2lwcGVkIiwgc3RhZ2Vfc3RhcnRlZCwgInJlcXVpcmVkIHByZXByb2Nlc3Npbmcgc3RhZ2UgZmFpbGVkIgogICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICByZXN1bHQgPSBOb25lCiAgICAgICAgICAgICAgICAgICAgICAgIHJldHJpZXMgPSAwCiAgICAgICAgICAgICAgICAgICAgICAgIHByZWRpY3Rpb24gPSB7fQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSgzKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdCA9IGFuYWx5emUoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW1hZ2VfYnl0ZXMoYnJhbmNoX2ltYWdlKSwgZW1vY2E9ZW1vY2EsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5naW5lX21vZGU9InJlYWwiLCBtb2RlbF9pZHM9WyJtYTNkX3JhZmRiIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBwcmVkaWN0aW9uID0gZ2V0X21hM2RfcHJlZGljdGlvbihyZXN1bHQpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBwcmVkaWN0aW9uLmdldCgib2siKSBvciAic2VydmljZV91bmF2YWlsYWJsZSIgbm90IGluIHByZWRpY3Rpb24uZ2V0KCJlcnJvciIsICIiKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgICAgICAgICAgICAgcmV0cmllcyArPSAxCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBhdHRlbXB0IDwgMjoKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKDUpCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1siZW1vY2FfcmV0cmllcyJdICs9IHJldHJpZXMKICAgICAgICAgICAgICAgICAgICAgICAgcm93WyJ2YXJpYW50cyJdW3ZhcmlhbnRdID0gcHJlZGljdGlvbgogICAgICAgICAgICAgICAgICAgICAgICBpZiBwcmVkaWN0aW9uLmdldCgib2siKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bc3RhZ2VfbmFtZV0gPSBzdGFnZV9yZWNvcmQoIm9rIiwgc3RhZ2Vfc3RhcnRlZCkKICAgICAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bc3RhZ2VfbmFtZV0gPSBzdGFnZV9yZWNvcmQoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImVycm9yIiwgc3RhZ2Vfc3RhcnRlZCwgcHJlZGljdGlvbi5nZXQoImVycm9yIiwgInVua25vd24gTUEzRCBlcnJvciIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgICAgICAgICAgICAgICAgIGVycm9yID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sidmFyaWFudHMiXVt2YXJpYW50XSA9IHsib2siOiBGYWxzZSwgImVycm9yIjogZXJyb3J9CiAgICAgICAgICAgICAgICAgICAgICAgIHJvd1sic3RhZ2VzIl1bc3RhZ2VfbmFtZV0gPSBzdGFnZV9yZWNvcmQoImVycm9yIiwgc3RhZ2Vfc3RhcnRlZCwgZXJyb3IpCgogICAgICAgICAgICAgICAgcm93WyJlbGFwc2VkX3NlY29uZHMiXSA9IHJvdW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSBpdGVtX3N0YXJ0ZWQsIDMpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOgogICAgICAgICAgICAgICAgcm93WyJmYXRhbF9lcnJvciJdID0gZiJ7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iCiAgICAgICAgICAgICAgICByb3dbImVsYXBzZWRfc2Vjb25kcyJdID0gcm91bmQodGltZS5wZXJmX2NvdW50ZXIoKSAtIGl0ZW1fc3RhcnRlZCwgMykKICAgICAgICAgICAgICAgIGZvciB2YXJpYW50IGluIFZBUklBTlRTOgogICAgICAgICAgICAgICAgICAgIHJvd1sidmFyaWFudHMiXS5zZXRkZWZhdWx0KHZhcmlhbnQsIHsib2siOiBGYWxzZSwgImVycm9yIjogcm93WyJmYXRhbF9lcnJvciJdfSkKCiAgICAgICAgICAgIHN0cmVhbS53cml0ZShqc29uLmR1bXBzKHJvdywgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpCiAgICAgICAgICAgIHN0cmVhbS5mbHVzaCgpCiAgICAgICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKICAgICAgICAgICAgaWYgbG9jYWxfaW5kZXggJSAyNSA9PSAwIG9yIGxvY2FsX2luZGV4ID09IGxlbihwZW5kaW5nKToKICAgICAgICAgICAgICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAogICAgICAgICAgICAgICAgcGVyX2ltYWdlID0gZWxhcHNlZCAvIG1heCgxLCBsb2NhbF9pbmRleCkKICAgICAgICAgICAgICAgIHZhbHVlcyA9ICIgIi5qb2luKAogICAgICAgICAgICAgICAgICAgIGYie25hbWV9Ontyb3dbJ3ZhcmlhbnRzJ10uZ2V0KG5hbWUsIHt9KS5nZXQoJ2luZGV4JywgJ0VSUicpfSIKICAgICAgICAgICAgICAgICAgICBmb3IgbmFtZSBpbiBWQVJJQU5UUwogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgcHJpbnQoCiAgICAgICAgICAgICAgICAgICAgZiJzaGFyZD17YXJncy5zaGFyZF9pbmRleH0gcHJvZ3Jlc3M9e2xvY2FsX2luZGV4fS97bGVuKHBlbmRpbmcpfSAiCiAgICAgICAgICAgICAgICAgICAgZiJmaWxlPXtmaWxlbmFtZX0ge3ZhbHVlc30gaXRlbT17cm93WydlbGFwc2VkX3NlY29uZHMnXTouMmZ9cyAiCiAgICAgICAgICAgICAgICAgICAgZiJyYXRlPXtwZXJfaW1hZ2U6LjJmfXMvaW1hZ2UiLAogICAgICAgICAgICAgICAgICAgIGZsdXNoPVRydWUsCiAgICAgICAgICAgICAgICApCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkK'))
(PREPROCESS_SCRIPTS / 'summarize_predata.py').write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IHNodXRpbApmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHN0YXRpc3RpY3MgaW1wb3J0IG1lYW4KCgpWQVJJQU5UUyA9ICgKICAgICJNQTNEUmVDb19ub1lPTE8iLAogICAgIk1BM0RSZV9ub1lPTE8iLAogICAgIk1BM0RDb19ub1lPTE8iLAopClZBUklBTlRfTEFCRUxTID0gewogICAgIk1BM0RSZUNvX25vWU9MTyI6ICJSZWFsRVNSR0FOICsgQ29kZUZvcm1lciArIE1BM0QiLAogICAgIk1BM0RSZV9ub1lPTE8iOiAiUmVhbEVTUkdBTiArIE1BM0QiLAogICAgIk1BM0RDb19ub1lPTE8iOiAiQ29kZUZvcm1lciArIE1BM0QiLAp9ClZBUklBTlRfRElSUyA9IHsKICAgICJNQTNEUmVDb19ub1lPTE8iOiAiUmVhbEVTUkdBTl9Db2RlRm9ybWVyX01BM0QiLAogICAgIk1BM0RSZV9ub1lPTE8iOiAiUmVhbEVTUkdBTl9NQTNEIiwKICAgICJNQTNEQ29fbm9ZT0xPIjogIkNvZGVGb3JtZXJfTUEzRCIsCn0KU1RBR0VTID0gKAogICAgInJlYWxlc3JnYW4iLAogICAgImNvZGVmb3JtZXJfcmF3IiwKICAgICJjb2RlZm9ybWVyX2FmdGVyX3JlYWxlc3JnYW4iLAogICAgImVtb2NhX21hM2RfcmVhbGVzcmdhbl9jb2RlZm9ybWVyIiwKICAgICJlbW9jYV9tYTNkX3JlYWxlc3JnYW4iLAogICAgImVtb2NhX21hM2RfY29kZWZvcm1lciIsCikKTEFCRUxTID0gWyJTdXJwcmlzZSIsICJGZWFyIiwgIkRpc2d1c3QiLCAiSGFwcGluZXNzIiwgIlNhZG5lc3MiLCAiQW5nZXIiLCAiTmV1dHJhbCJdCgoKZGVmIGxvYWRfbGF0ZXN0X3Jvd3MocmVzdWx0c19kaXI6IFBhdGgpIC0+IGRpY3RbdHVwbGVbaW50LCBzdHJdLCBkaWN0XToKICAgIHJvd3M6IGRpY3RbdHVwbGVbaW50LCBzdHJdLCBkaWN0XSA9IHt9CiAgICBmaWxlcyA9IHNvcnRlZChyZXN1bHRzX2Rpci5nbG9iKCJzaGFyZF8qLmpzb25sIikpCiAgICBpZiBub3QgZmlsZXM6CiAgICAgICAgcmFpc2UgRmlsZU5vdEZvdW5kRXJyb3IoZiJObyBzaGFyZF8qLmpzb25sIGZpbGVzIGluIHtyZXN1bHRzX2Rpcn0iKQogICAgZm9yIHBhdGggaW4gZmlsZXM6CiAgICAgICAgd2l0aCBwYXRoLm9wZW4oZW5jb2Rpbmc9InV0Zi04IikgYXMgc3RyZWFtOgogICAgICAgICAgICBmb3IgbGluZSBpbiBzdHJlYW06CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgcm93ID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgICAgIHJvd3NbKGludChyb3dbImxhYmVsIl0pLCBzdHIocm93WyJmaWxlIl0pKV0gPSByb3cKICAgICAgICAgICAgICAgIGV4Y2VwdCAoanNvbi5KU09ORGVjb2RlRXJyb3IsIEtleUVycm9yLCBUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICByZXR1cm4gcm93cwoKCmRlZiB2YWxpZF9wcmVkaWN0aW9uKHJvdzogZGljdCwgdmFyaWFudDogc3RyKSAtPiBkaWN0IHwgTm9uZToKICAgIHByZWRpY3Rpb24gPSByb3cuZ2V0KCJ2YXJpYW50cyIsIHt9KS5nZXQodmFyaWFudCwge30pCiAgICBpZiBwcmVkaWN0aW9uLmdldCgib2siKSBhbmQgbGVuKHByZWRpY3Rpb24uZ2V0KCJwcm9icyIsIFtdKSkgPT0gNzoKICAgICAgICByZXR1cm4gcHJlZGljdGlvbgogICAgcmV0dXJuIE5vbmUKCgpkZWYgY2FsY3VsYXRlX21ldHJpY3Mocm93czogbGlzdFtkaWN0XSwgdmFyaWFudDogc3RyLCBleHBlY3RlZDogaW50KSAtPiBkaWN0OgogICAgdmFsaWQgPSBbKHJvdywgdmFsaWRfcHJlZGljdGlvbihyb3csIHZhcmlhbnQpKSBmb3Igcm93IGluIHJvd3NdCiAgICB2YWxpZCA9IFsocm93LCBwcmVkKSBmb3Igcm93LCBwcmVkIGluIHZhbGlkIGlmIHByZWQgaXMgbm90IE5vbmVdCiAgICBjb25mdXNpb24gPSBbWzAgZm9yIF8gaW4gTEFCRUxTXSBmb3IgXyBpbiBMQUJFTFNdCiAgICBmb3Igcm93LCBwcmVkaWN0aW9uIGluIHZhbGlkOgogICAgICAgIHRydWVfaW5kZXggPSBpbnQocm93WyJsYWJlbCJdKSAtIDEKICAgICAgICBwcmVkX2luZGV4ID0gaW50KHByZWRpY3Rpb25bImluZGV4Il0pCiAgICAgICAgaWYgdHJ1ZV9pbmRleCBpbiByYW5nZSg3KSBhbmQgcHJlZF9pbmRleCBpbiByYW5nZSg3KToKICAgICAgICAgICAgY29uZnVzaW9uW3RydWVfaW5kZXhdW3ByZWRfaW5kZXhdICs9IDEKCiAgICBwZXJfY2xhc3MgPSBbXQogICAgZjFfdmFsdWVzID0gW10KICAgIGZvciBpbmRleCwgbmFtZSBpbiBlbnVtZXJhdGUoTEFCRUxTKToKICAgICAgICB0cCA9IGNvbmZ1c2lvbltpbmRleF1baW5kZXhdCiAgICAgICAgc3VwcG9ydCA9IHN1bShjb25mdXNpb25baW5kZXhdKQogICAgICAgIHByZWRpY3RlZCA9IHN1bShjb25mdXNpb25bcm93XVtpbmRleF0gZm9yIHJvdyBpbiByYW5nZSg3KSkKICAgICAgICBwcmVjaXNpb24gPSB0cCAvIHByZWRpY3RlZCBpZiBwcmVkaWN0ZWQgZWxzZSAwLjAKICAgICAgICByZWNhbGwgPSB0cCAvIHN1cHBvcnQgaWYgc3VwcG9ydCBlbHNlIDAuMAogICAgICAgIGYxID0gMiAqIHByZWNpc2lvbiAqIHJlY2FsbCAvIChwcmVjaXNpb24gKyByZWNhbGwpIGlmIHByZWNpc2lvbiArIHJlY2FsbCBlbHNlIDAuMAogICAgICAgIGlmIHN1cHBvcnQ6CiAgICAgICAgICAgIGYxX3ZhbHVlcy5hcHBlbmQoZjEgKiAxMDApCiAgICAgICAgcGVyX2NsYXNzLmFwcGVuZCh7CiAgICAgICAgICAgICJsYWJlbCI6IG5hbWUsCiAgICAgICAgICAgICJzdXBwb3J0Ijogc3VwcG9ydCwKICAgICAgICAgICAgInByZWNpc2lvbl9wY3QiOiBwcmVjaXNpb24gKiAxMDAsCiAgICAgICAgICAgICJyZWNhbGxfcGN0IjogcmVjYWxsICogMTAwIGlmIHN1cHBvcnQgZWxzZSBOb25lLAogICAgICAgICAgICAiZjFfcGN0IjogZjEgKiAxMDAgaWYgc3VwcG9ydCBlbHNlIE5vbmUsCiAgICAgICAgfSkKCiAgICBjb3JyZWN0ID0gc3VtKGNvbmZ1c2lvbltpXVtpXSBmb3IgaSBpbiByYW5nZSg3KSkKICAgIHdyb25nID0gbGVuKHZhbGlkKSAtIGNvcnJlY3QKICAgIGZhaWxlZCA9IGV4cGVjdGVkIC0gbGVuKHZhbGlkKQogICAgY29uZmlkZW5jZXMgPSBbZmxvYXQocHJlZFsiY29uZmlkZW5jZSJdKSBmb3IgXywgcHJlZCBpbiB2YWxpZF0KICAgIHJldHVybiB7CiAgICAgICAgInZhcmlhbnQiOiB2YXJpYW50LAogICAgICAgICJkZXNjcmlwdGlvbiI6IFZBUklBTlRfTEFCRUxTW3ZhcmlhbnRdLAogICAgICAgICJuX3Rlc3QiOiBleHBlY3RlZCwKICAgICAgICAibl92YWxpZCI6IGxlbih2YWxpZCksCiAgICAgICAgIm5fZmFpbGVkIjogZmFpbGVkLAogICAgICAgICJuX3dyb25nX3ZhbGlkX3ByZWRpY3Rpb25zIjogd3JvbmcsCiAgICAgICAgIm5fY29ycmVjdCI6IGNvcnJlY3QsCiAgICAgICAgImNvdmVyYWdlX3BjdCI6IDEwMCAqIGxlbih2YWxpZCkgLyBleHBlY3RlZCBpZiBleHBlY3RlZCBlbHNlIE5vbmUsCiAgICAgICAgImFjY3VyYWN5X3ZhbGlkX3BjdCI6IDEwMCAqIGNvcnJlY3QgLyBsZW4odmFsaWQpIGlmIHZhbGlkIGVsc2UgTm9uZSwKICAgICAgICAiYWNjdXJhY3lfYWxsX3BjdCI6IDEwMCAqIGNvcnJlY3QgLyBleHBlY3RlZCBpZiBleHBlY3RlZCBlbHNlIE5vbmUsCiAgICAgICAgIm1hY3JvX2YxX3ZhbGlkX3BjdCI6IG1lYW4oZjFfdmFsdWVzKSBpZiBmMV92YWx1ZXMgZWxzZSBOb25lLAogICAgICAgICJtZWFuX2NvbmZpZGVuY2VfdmFsaWRfcGN0IjogMTAwICogbWVhbihjb25maWRlbmNlcykgaWYgY29uZmlkZW5jZXMgZWxzZSBOb25lLAogICAgICAgICJwZXJfY2xhc3MiOiBwZXJfY2xhc3MsCiAgICAgICAgImNvbmZ1c2lvbl9tYXRyaXhfdHJ1ZV9yb3dzX3ByZWRpY3RlZF9jb2x1bW5zIjogY29uZnVzaW9uLAogICAgfQoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlN1bW1hcml6ZSB0aGUgdGhyZWUgbm8tWU9MTyBwcmVwcm9jZXNzaW5nIGFibGF0aW9ucy4iKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1idW5kbGUtcm9vdCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcmVzdWx0cy1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWV4YW1wbGVzLXBlci1tb2RlbCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1leHBlY3RlZC1pbWFnZXMiLCB0eXBlPWludCwgZGVmYXVsdD0wLAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJFeHBlY3RlZCBkZW5vbWluYXRvciBmb3IgYSBzbW9rZSBzdWJzZXQ7IGRlZmF1bHRzIHRvIHRoZSBmdWxsIHRlc3Qgc2V0LiIpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgIGJ1bmRsZSA9IGFyZ3MuYnVuZGxlX3Jvb3QucmVzb2x2ZSgpCiAgICByZXN1bHRzX2RpciA9IGFyZ3MucmVzdWx0c19kaXIucmVzb2x2ZSgpCiAgICByb3dzX2J5X2tleSA9IGxvYWRfbGF0ZXN0X3Jvd3MocmVzdWx0c19kaXIpCiAgICB3aXRoIChidW5kbGUgLyAiZGF0YXNldCIgLyAidGVzdF9sYWJlbHMuY3N2Iikub3BlbihlbmNvZGluZz0idXRmLTgtc2lnIiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIGxhYmVscyA9IHtyb3dbImltYWdlIl06IGludChyb3dbImxhYmVsIl0pIGZvciByb3cgaW4gY3N2LkRpY3RSZWFkZXIoc3RyZWFtKX0KICAgIG9yZGVyZWRfcm93cyA9IFsKICAgICAgICByb3dzX2J5X2tleVsobGFiZWwsIGZpbGVuYW1lKV0KICAgICAgICBmb3IgbGFiZWwsIGZpbGVuYW1lIGluIHNvcnRlZCgoKHZhbHVlLCBuYW1lKSBmb3IgbmFtZSwgdmFsdWUgaW4gbGFiZWxzLml0ZW1zKCkpLCBrZXk9bGFtYmRhIHg6ICh4WzBdLCB4WzFdKSkKICAgICAgICBpZiAobGFiZWwsIGZpbGVuYW1lKSBpbiByb3dzX2J5X2tleQogICAgXQogICAgZXhwZWN0ZWQgPSBhcmdzLmV4cGVjdGVkX2ltYWdlcyBvciBsZW4obGFiZWxzKQogICAgaWYgZXhwZWN0ZWQgPCAxIG9yIGV4cGVjdGVkID4gbGVuKGxhYmVscyk6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIi0tZXhwZWN0ZWQtaW1hZ2VzIG11c3QgYmUgYmV0d2VlbiAxIGFuZCB7bGVuKGxhYmVscyl9IikKICAgIG1ldHJpY3MgPSB7dmFyaWFudDogY2FsY3VsYXRlX21ldHJpY3Mob3JkZXJlZF9yb3dzLCB2YXJpYW50LCBleHBlY3RlZCkgZm9yIHZhcmlhbnQgaW4gVkFSSUFOVFN9CgogICAgc3VtbWFyeV9wYXRoID0gcmVzdWx0c19kaXIgLyAic3VtbWFyeS5qc29uIgogICAgc3VtbWFyeV9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyh7CiAgICAgICAgImRhdGFzZXQiOiAiUkFGLURCIGFsaWduZWQgdGVzdCBpbWFnZXMiLAogICAgICAgICJwcm9jZXNzZWRfcm93cyI6IGxlbihvcmRlcmVkX3Jvd3MpLAogICAgICAgICJleHBlY3RlZF9yb3dzIjogZXhwZWN0ZWQsCiAgICAgICAgImFsbF9pbWFnZXNfcHJvY2Vzc2VkIjogbGVuKG9yZGVyZWRfcm93cykgPT0gZXhwZWN0ZWQsCiAgICAgICAgImZsb3dzIjoge3ZhcmlhbnQ6IFZBUklBTlRfTEFCRUxTW3ZhcmlhbnRdIGZvciB2YXJpYW50IGluIFZBUklBTlRTfSwKICAgICAgICAibWV0cmljcyI6IG1ldHJpY3MsCiAgICAgICAgImVycm9yX2ltYWdlcyI6IHsKICAgICAgICAgICAgdmFyaWFudDogewogICAgICAgICAgICAgICAgImNvdW50IjogbWV0cmljc1t2YXJpYW50XVsibl93cm9uZ192YWxpZF9wcmVkaWN0aW9ucyJdLAogICAgICAgICAgICAgICAgInNlbGVjdGlvbiI6IGYiVG9wIHthcmdzLmV4YW1wbGVzX3Blcl9tb2RlbH0gd3JvbmcgcHJlZGljdGlvbnMgYnkgY29uZmlkZW5jZSwgZGVzY2VuZGluZyIsCiAgICAgICAgICAgICAgICAiZGlyZWN0b3J5Ijogc3RyKHJlc3VsdHNfZGlyIC8gIm1pc2NsYXNzaWZpZWQiIC8gVkFSSUFOVF9ESVJTW3ZhcmlhbnRdKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBmb3IgdmFyaWFudCBpbiBWQVJJQU5UUwogICAgICAgIH0sCiAgICAgICAgIm5vdGUiOiAiRXhpc3RpbmcgUE9TVEVSL01BM0QvWU9MTy1iYXNlZCByZXN1bHRzIHdlcmUgbm90IHJlcnVuIG9yIG92ZXJ3cml0dGVuLiIsCiAgICB9LCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0yKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCgogICAgbWV0cmljX3BhdGggPSByZXN1bHRzX2RpciAvICJtZXRyaWNzLmNzdiIKICAgIG1ldHJpY19jb2x1bW5zID0gWwogICAgICAgICJ2YXJpYW50IiwgImRlc2NyaXB0aW9uIiwgIm5fdGVzdCIsICJuX3ZhbGlkIiwgIm5fZmFpbGVkIiwgIm5fd3JvbmdfdmFsaWRfcHJlZGljdGlvbnMiLAogICAgICAgICJuX2NvcnJlY3QiLCAiY292ZXJhZ2VfcGN0IiwgImFjY3VyYWN5X3ZhbGlkX3BjdCIsICJhY2N1cmFjeV9hbGxfcGN0IiwKICAgICAgICAibWFjcm9fZjFfdmFsaWRfcGN0IiwgIm1lYW5fY29uZmlkZW5jZV92YWxpZF9wY3QiLAogICAgXQogICAgd2l0aCBtZXRyaWNfcGF0aC5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04LXNpZyIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihzdHJlYW0sIGZpZWxkbmFtZXM9bWV0cmljX2NvbHVtbnMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3IgdmFyaWFudCBpbiBWQVJJQU5UUzoKICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KHtrZXk6IG1ldHJpY3NbdmFyaWFudF1ba2V5XSBmb3Iga2V5IGluIG1ldHJpY19jb2x1bW5zfSkKCiAgICBzdGFnZV9jb3VudHM6IGRpY3Rbc3RyLCBDb3VudGVyXSA9IHtzdGFnZTogQ291bnRlcigpIGZvciBzdGFnZSBpbiBTVEFHRVN9CiAgICBzdGFnZV9lcnJvcnM6IGRpY3Rbc3RyLCBDb3VudGVyXSA9IHtzdGFnZTogQ291bnRlcigpIGZvciBzdGFnZSBpbiBTVEFHRVN9CiAgICBmb3Igcm93IGluIG9yZGVyZWRfcm93czoKICAgICAgICBmb3Igc3RhZ2UgaW4gU1RBR0VTOgogICAgICAgICAgICByZWNvcmQgPSByb3cuZ2V0KCJzdGFnZXMiLCB7fSkuZ2V0KHN0YWdlKQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyZWNvcmQsIGRpY3QpOgogICAgICAgICAgICAgICAgc3RhZ2VfY291bnRzW3N0YWdlXVsibWlzc2luZyJdICs9IDEKICAgICAgICAgICAgICAgIHN0YWdlX2Vycm9yc1tzdGFnZV1bInN0YWdlIHJlY29yZCBtaXNzaW5nIl0gKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc3RhdHVzID0gc3RyKHJlY29yZC5nZXQoInN0YXR1cyIsICJtaXNzaW5nIikpCiAgICAgICAgICAgIHN0YWdlX2NvdW50c1tzdGFnZV1bc3RhdHVzXSArPSAxCiAgICAgICAgICAgIGlmIHN0YXR1cyA9PSAiZXJyb3IiOgogICAgICAgICAgICAgICAgc3RhZ2VfZXJyb3JzW3N0YWdlXVtzdHIocmVjb3JkLmdldCgiZXJyb3IiKSBvciAidW5zcGVjaWZpZWQgZXJyb3IiKV0gKz0gMQoKICAgIHN0YWdlX2NzdiA9IHJlc3VsdHNfZGlyIC8gInN0YWdlX2Vycm9ycy5jc3YiCiAgICB3aXRoIHN0YWdlX2Nzdi5vcGVuKCJ3IiwgZW5jb2Rpbmc9InV0Zi04LXNpZyIsIG5ld2xpbmU9IiIpIGFzIHN0cmVhbToKICAgICAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihzdHJlYW0sIGZpZWxkbmFtZXM9WyJzdGFnZSIsICJzdGF0dXMiLCAiY291bnQiLCAiZXJyb3IiXSkKICAgICAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgICAgIGZvciBzdGFnZSBpbiBTVEFHRVM6CiAgICAgICAgICAgIGZvciBzdGF0dXMsIGNvdW50IGluIHNvcnRlZChzdGFnZV9jb3VudHNbc3RhZ2VdLml0ZW1zKCkpOgogICAgICAgICAgICAgICAgZXJyb3JzID0gc3RhZ2VfZXJyb3JzW3N0YWdlXQogICAgICAgICAgICAgICAgaWYgc3RhdHVzID09ICJlcnJvciIgYW5kIGVycm9yczoKICAgICAgICAgICAgICAgICAgICBmb3IgZXJyb3IsIGVycm9yX2NvdW50IGluIGVycm9ycy5tb3N0X2NvbW1vbigpOgogICAgICAgICAgICAgICAgICAgICAgICB3cml0ZXIud3JpdGVyb3coeyJzdGFnZSI6IHN0YWdlLCAic3RhdHVzIjogc3RhdHVzLCAiY291bnQiOiBlcnJvcl9jb3VudCwgImVycm9yIjogZXJyb3J9KQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICB3cml0ZXIud3JpdGVyb3coeyJzdGFnZSI6IHN0YWdlLCAic3RhdHVzIjogc3RhdHVzLCAiY291bnQiOiBjb3VudCwgImVycm9yIjogIiJ9KQoKICAgIHByZWRpY3Rpb25zX3BhdGggPSByZXN1bHRzX2RpciAvICJwcmVkaWN0aW9ucy5jc3YiCiAgICBwcmVkaWN0aW9uX2NvbHVtbnMgPSBbImxhYmVsIiwgImxhYmVsX25hbWUiLCAiZmlsZSJdCiAgICBmb3IgdmFyaWFudCBpbiBWQVJJQU5UUzoKICAgICAgICBwcmVkaWN0aW9uX2NvbHVtbnMuZXh0ZW5kKFtmInt2YXJpYW50fV92YWxpZCIsIGYie3ZhcmlhbnR9X3ByZWRpY3RlZCIsIGYie3ZhcmlhbnR9X2NvbmZpZGVuY2UiLCBmInt2YXJpYW50fV9lcnJvciJdKQogICAgd2l0aCBwcmVkaWN0aW9uc19wYXRoLm9wZW4oInciLCBlbmNvZGluZz0idXRmLTgtc2lnIiwgbmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1wcmVkaWN0aW9uX2NvbHVtbnMpCiAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICBmb3Igcm93IGluIG9yZGVyZWRfcm93czoKICAgICAgICAgICAgbGFiZWwgPSBpbnQocm93WyJsYWJlbCJdKQogICAgICAgICAgICBmbGF0ID0geyJsYWJlbCI6IGxhYmVsLCAibGFiZWxfbmFtZSI6IExBQkVMU1tsYWJlbCAtIDFdLCAiZmlsZSI6IHJvd1siZmlsZSJdfQogICAgICAgICAgICBmb3IgdmFyaWFudCBpbiBWQVJJQU5UUzoKICAgICAgICAgICAgICAgIHByZWRpY3Rpb24gPSByb3cuZ2V0KCJ2YXJpYW50cyIsIHt9KS5nZXQodmFyaWFudCwge30pCiAgICAgICAgICAgICAgICBmbGF0W2Yie3ZhcmlhbnR9X3ZhbGlkIl0gPSBib29sKHByZWRpY3Rpb24uZ2V0KCJvayIpKQogICAgICAgICAgICAgICAgZmxhdFtmInt2YXJpYW50fV9wcmVkaWN0ZWQiXSA9IExBQkVMU1tpbnQocHJlZGljdGlvblsiaW5kZXgiXSldIGlmIHByZWRpY3Rpb24uZ2V0KCJvayIpIGVsc2UgIiIKICAgICAgICAgICAgICAgIGZsYXRbZiJ7dmFyaWFudH1fY29uZmlkZW5jZSJdID0gcHJlZGljdGlvbi5nZXQoImNvbmZpZGVuY2UiLCAiIikKICAgICAgICAgICAgICAgIGZsYXRbZiJ7dmFyaWFudH1fZXJyb3IiXSA9IHByZWRpY3Rpb24uZ2V0KCJlcnJvciIsICIiKQogICAgICAgICAgICB3cml0ZXIud3JpdGVyb3coZmxhdCkKCiAgICBtaXNjbGFzc2lmaWVkX3Jvb3QgPSByZXN1bHRzX2RpciAvICJtaXNjbGFzc2lmaWVkIgogICAgZm9yIHZhcmlhbnQgaW4gVkFSSUFOVFM6CiAgICAgICAgb3V0cHV0X2RpciA9IG1pc2NsYXNzaWZpZWRfcm9vdCAvIFZBUklBTlRfRElSU1t2YXJpYW50XQogICAgICAgIGlmIG91dHB1dF9kaXIuZXhpc3RzKCk6CiAgICAgICAgICAgIHNodXRpbC5ybXRyZWUob3V0cHV0X2RpcikKICAgICAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICB3cm9uZ19yb3dzID0gW10KICAgICAgICBmb3Igcm93IGluIG9yZGVyZWRfcm93czoKICAgICAgICAgICAgcHJlZGljdGlvbiA9IHZhbGlkX3ByZWRpY3Rpb24ocm93LCB2YXJpYW50KQogICAgICAgICAgICBpZiBwcmVkaWN0aW9uIGlzIG5vdCBOb25lIGFuZCBpbnQocHJlZGljdGlvblsiaW5kZXgiXSkgIT0gaW50KHJvd1sibGFiZWwiXSkgLSAxOgogICAgICAgICAgICAgICAgd3Jvbmdfcm93cy5hcHBlbmQoKHJvdywgcHJlZGljdGlvbikpCiAgICAgICAgd3Jvbmdfcm93cy5zb3J0KGtleT1sYW1iZGEgaXRlbTogKC1mbG9hdChpdGVtWzFdWyJjb25maWRlbmNlIl0pLCBzdHIoaXRlbVswXVsiZmlsZSJdKSkpCiAgICAgICAgc2VsZWN0ZWQgPSB3cm9uZ19yb3dzWzptYXgoMCwgYXJncy5leGFtcGxlc19wZXJfbW9kZWwpXQogICAgICAgIG1ldGFkYXRhX3BhdGggPSBvdXRwdXRfZGlyIC8gIm1ldGFkYXRhLmNzdiIKICAgICAgICB3aXRoIG1ldGFkYXRhX3BhdGgub3BlbigidyIsIGVuY29kaW5nPSJ1dGYtOC1zaWciLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1bCiAgICAgICAgICAgICAgICAicmFuayIsICJmaWxlIiwgInRydWVfbGFiZWwiLCAicHJlZGljdGVkX2xhYmVsIiwgImNvbmZpZGVuY2UiLCAicHJvYmFiaWxpdHlfdHJ1ZV9sYWJlbCIsCiAgICAgICAgICAgIF0pCiAgICAgICAgICAgIHdyaXRlci53cml0ZWhlYWRlcigpCiAgICAgICAgICAgIGZvciByYW5rLCAocm93LCBwcmVkaWN0aW9uKSBpbiBlbnVtZXJhdGUoc2VsZWN0ZWQsIDEpOgogICAgICAgICAgICAgICAgdHJ1ZV9sYWJlbCA9IGludChyb3dbImxhYmVsIl0pCiAgICAgICAgICAgICAgICBwcmVkaWN0ZWRfbGFiZWwgPSBpbnQocHJlZGljdGlvblsiaW5kZXgiXSkgKyAxCiAgICAgICAgICAgICAgICBzb3VyY2UgPSBidW5kbGUgLyAiZGF0YXNldCIgLyAiREFUQVNFVCIgLyAidGVzdCIgLyBzdHIodHJ1ZV9sYWJlbCkgLyBzdHIocm93WyJmaWxlIl0pCiAgICAgICAgICAgICAgICBkZXN0aW5hdGlvbl9uYW1lID0gKAogICAgICAgICAgICAgICAgICAgIGYie3Jhbms6MDJkfV90cnVlLXtMQUJFTFNbdHJ1ZV9sYWJlbCAtIDFdfV9wcmVkLXtMQUJFTFNbcHJlZGljdGVkX2xhYmVsIC0gMV19XyIKICAgICAgICAgICAgICAgICAgICBmImNvbmYte2Zsb2F0KHByZWRpY3Rpb25bJ2NvbmZpZGVuY2UnXSk6LjNmfV97UGF0aChyb3dbJ2ZpbGUnXSkubmFtZX0iCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBpZiBzb3VyY2UuaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgICAgIHNodXRpbC5jb3B5Mihzb3VyY2UsIG91dHB1dF9kaXIgLyBkZXN0aW5hdGlvbl9uYW1lKQogICAgICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93KHsKICAgICAgICAgICAgICAgICAgICAicmFuayI6IHJhbmssCiAgICAgICAgICAgICAgICAgICAgImZpbGUiOiByb3dbImZpbGUiXSwKICAgICAgICAgICAgICAgICAgICAidHJ1ZV9sYWJlbCI6IExBQkVMU1t0cnVlX2xhYmVsIC0gMV0sCiAgICAgICAgICAgICAgICAgICAgInByZWRpY3RlZF9sYWJlbCI6IExBQkVMU1twcmVkaWN0ZWRfbGFiZWwgLSAxXSwKICAgICAgICAgICAgICAgICAgICAiY29uZmlkZW5jZSI6IHByZWRpY3Rpb25bImNvbmZpZGVuY2UiXSwKICAgICAgICAgICAgICAgICAgICAicHJvYmFiaWxpdHlfdHJ1ZV9sYWJlbCI6IHByZWRpY3Rpb25bInByb2JzIl1bdHJ1ZV9sYWJlbCAtIDFdLAogICAgICAgICAgICAgICAgfSkKCiAgICBwcmludChmIlByb2Nlc3NlZCByb3dzOiB7bGVuKG9yZGVyZWRfcm93cyl9L3tleHBlY3RlZH0iKQogICAgcHJpbnQoIlN0YWdlIHN0YXR1cy9lcnJvciBjb3VudHM6IikKICAgIGZvciBzdGFnZSBpbiBTVEFHRVM6CiAgICAgICAgc3RhdHVzX2NvdW50cyA9ICIsICIuam9pbihmIntzdGF0dXN9PXtjb3VudH0iIGZvciBzdGF0dXMsIGNvdW50IGluIHNvcnRlZChzdGFnZV9jb3VudHNbc3RhZ2VdLml0ZW1zKCkpKQogICAgICAgIHByaW50KGYiICB7c3RhZ2V9OiB7c3RhdHVzX2NvdW50cyBvciAnbm8gcmVjb3Jkcyd9IikKICAgICAgICBmb3IgZXJyb3IsIGNvdW50IGluIHN0YWdlX2Vycm9yc1tzdGFnZV0ubW9zdF9jb21tb24oNSk6CiAgICAgICAgICAgIHByaW50KGYiICAgIHtjb3VudH0gw5cge2Vycm9yfSIpCiAgICBwcmludCgiTW9kZWwgcmVzdWx0czoiKQogICAgZm9yIHZhcmlhbnQgaW4gVkFSSUFOVFM6CiAgICAgICAgcmVzdWx0ID0gbWV0cmljc1t2YXJpYW50XQogICAgICAgIHByaW50KAogICAgICAgICAgICBmIiAge3ZhcmlhbnR9OiB3cm9uZz17cmVzdWx0WyduX3dyb25nX3ZhbGlkX3ByZWRpY3Rpb25zJ119LCBmYWlsZWQ9e3Jlc3VsdFsnbl9mYWlsZWQnXX0sICIKICAgICAgICAgICAgZiJjb3ZlcmFnZT17cmVzdWx0Wydjb3ZlcmFnZV9wY3QnXTouMmZ9JSwgYWNjdXJhY3kodmFsaWQpPXtyZXN1bHRbJ2FjY3VyYWN5X3ZhbGlkX3BjdCddfSwgIgogICAgICAgICAgICBmImFjY3VyYWN5KGFsbCk9e3Jlc3VsdFsnYWNjdXJhY3lfYWxsX3BjdCddfSUsIG1hY3JvX2YxPXtyZXN1bHRbJ21hY3JvX2YxX3ZhbGlkX3BjdCddfSUiCiAgICAgICAgKQogICAgICAgIHByaW50KGYiICAgIFRvcCBtaXNjbGFzc2lmaWVkIGV4YW1wbGVzOiB7cmVzdWx0c19kaXIgLyAnbWlzY2xhc3NpZmllZCcgLyBWQVJJQU5UX0RJUlNbdmFyaWFudF19IikKICAgIHByaW50KGYiV3JvdGUge3N1bW1hcnlfcGF0aH0sIHttZXRyaWNfcGF0aH0sIHtzdGFnZV9jc3Z9LCBhbmQge3ByZWRpY3Rpb25zX3BhdGh9IikKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo='))

# Reuse the existing EMOCA service launcher, but point it at the new no-YOLO worker.
launcher_source = (SCRIPTS / 'launch_workers.py').read_text(encoding='utf-8')
old_worker_line = 'worker_script = Path(__file__).resolve().parent / "run_worker.py"'
if old_worker_line not in launcher_source:
    raise RuntimeError('Attached Dataset launch_workers.py has an unexpected version.')
launcher_source = launcher_source.replace(
    old_worker_line,
    'worker_script = Path(__file__).resolve().parent / "run_predata_worker.py"',
)
PREPROCESS_LAUNCHER = PREPROCESS_SCRIPTS / 'launch_predata_workers.py'
PREPROCESS_LAUNCHER.write_text(launcher_source, encoding='utf-8')
RESULTS = WORKDIR / 'results' / 'predata_no_yolo'
SMOKE = RESULTS / 'smoke'
run_quiet([sys.executable, str(PREPROCESS_LAUNCHER),
           '--bundle-root', str(BUNDLE_ROOT), '--workdir', str(WORKDIR),
           '--output-dir', str(SMOKE), '--limit', '2'],
          label='Two-image no-YOLO smoke inference')
subprocess.run([sys.executable, str(PREPROCESS_SCRIPTS / 'summarize_predata.py'),
                '--bundle-root', str(BUNDLE_ROOT), '--results-dir', str(SMOKE),
                '--examples-per-model', '0', '--expected-images', '2'], check=True)
smoke_summary = json.loads((SMOKE / 'summary.json').read_text(encoding='utf-8'))
unavailable = [name for name, result in smoke_summary['metrics'].items() if result['n_valid'] == 0]
if unavailable:
    raise RuntimeError('No-YOLO smoke produced no valid predictions for: ' + str(unavailable) + '; full test was not started.')
print('No-YOLO smoke passed; continuing to the full test.')


In [ ]:
# Full test: only the three new branches run; previous benchmark outputs are not overwritten.
FULL = RESULTS / 'full'
if RUN_FULL_BENCHMARK:
    run_quiet([sys.executable, str(PREPROCESS_LAUNCHER),
               '--bundle-root', str(BUNDLE_ROOT), '--workdir', str(WORKDIR),
               '--output-dir', str(FULL)],
              label='Full RAF-DB no-YOLO inference')
    subprocess.run([sys.executable, str(PREPROCESS_SCRIPTS / 'summarize_predata.py'),
                    '--bundle-root', str(BUNDLE_ROOT), '--results-dir', str(FULL),
                    '--examples-per-model', '10'], check=True)
